# Vision 핵심 코드 빈칸 문제집

출제 포인트와 직접 관련 있는 코드만 `???`로 치환했습니다. 정답 키는 각 코드 셀에 등장하는 `???` 순서와 동일합니다.


## 01_ResNet18_CIFAR10.ipynb

### 문제 1. 1) Dataset: CIFAR-10 로드 + 입력 크기 확인
- 원본 파일/셀: `01_ResNet18_CIFAR10.ipynb` / code cell index `5`
- 목적: 목적:** 학습/평가에 사용할 데이터셋을 준비합니다.
- 관찰 포인트:
  - **관찰 포인트**
  - CNN 입력은 **(C,H,W)** 텐서이며, CIFAR-10은 **32×32** 입니다.
  - augmentation(RandomCrop/Flip)을 적용합니다.
  - CIFAR-10 데이터셋 구성
- 출제 포인트:
  - 학습/평가 의도에 맞는 데이터 구성과 이미지 변형
- 빈칸 기준:
  - transform 순차 구성
  - random crop augmentation
  - horizontal flip augmentation
  - tensor 변환
  - 채널 정규화
  - transform 순차 구성
  - tensor 변환
  - 채널 정규화


In [ ]:
# CIFAR-10에서 널리 쓰는 normalize 값
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)  # 채널별 평균(R,G,B)
CIFAR10_STD  = (0.2023, 0.1994, 0.2010)  # 채널별 표준편차(R,G,B)

train_tfms = ???([  # train transform(augmentation 포함)
    ???,  # 32x32를 padding 후 random crop
    ???,  # 좌우 반전
    ???,  # PIL -> torch tensor (C,H,W), 0~1
    ???,  # 정규화, -2.5~2.5
])

test_tfms = ???([  # test transform(augmentation 없음)
    ???,  # 텐서 변환
    ???,  # 정규화
])

data_root = './data'  # 데이터 저장 경로
train_set = torchvision.datasets.CIFAR10(root=data_root, train=True, download=True, transform=train_tfms)  # train set
test_set  = torchvision.datasets.CIFAR10(root=data_root, train=False, download=True, transform=test_tfms)  # test set

class_names = train_set.classes  # 클래스 이름 목록
num_classes = len(class_names)  # 클래스 개수(10)
print('classes:', class_names)  # 클래스 출력

# 이미지 사이즈 출력
x0, y0 = train_set[0]  # 한 샘플 로드(이미 transform 적용된 텐서)
print('Sample image tensor shape (C,H,W):', tuple(x0.shape))  # (3,32,32)

batch_size = 256  # 배치 크기
num_workers = min(8, os.cpu_count() or 2)  # dataloader worker 수(환경에 맞게)

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)  # train loader
test_loader  = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)  # test loader

print('Total batch size (train: %d, test: %d)' % (len(train_loader), len(test_loader)))  # 배치 개수 확인


#### 정답

??? 순서: `T.Compose`, `T.RandomCrop(32, padding=4)`, `T.RandomHorizontalFlip()`, `T.ToTensor()`, `T.Normalize(CIFAR10_MEAN, CIFAR10_STD)`, `T.Compose`, `T.ToTensor()`, `T.Normalize(CIFAR10_MEAN, CIFAR10_STD)`

#### 간략 해설

1. `T.Compose`: 여러 transform을 지정한 순서대로 적용해야 하므로 사용합니다.
2. `T.RandomCrop(32, padding=4)`: 학습 이미지를 무작위로 자르고 padding을 주어 위치 변화에 덜 민감하게 만듭니다.
3. `T.RandomHorizontalFlip()`: 좌우 반전을 추가해 학습 데이터 다양성을 늘리는 augmentation입니다.
4. `T.ToTensor()`: PIL 이미지를 PyTorch가 처리하는 C,H,W tensor로 바꿉니다.
5. `T.Normalize(CIFAR10_MEAN, CIFAR10_STD)`: 채널별 평균과 표준편차로 입력 분포를 맞춰 학습을 안정화합니다.
6. `T.Compose`: 여러 transform을 지정한 순서대로 적용해야 하므로 사용합니다.
7. `T.ToTensor()`: PIL 이미지를 PyTorch가 처리하는 C,H,W tensor로 바꿉니다.
8. `T.Normalize(CIFAR10_MEAN, CIFAR10_STD)`: 채널별 평균과 표준편차로 입력 분포를 맞춰 학습을 안정화합니다.


### 문제 2. 2) ResNet18 모델 준비
- 원본 파일/셀: `01_ResNet18_CIFAR10.ipynb` / code cell index `8`
- 목적: 목적:** CIFAR-10(32×32)에 맞는 ResNet18을 구성합니다.
- 관찰 포인트:
  - **관찰 포인트**
  - ImageNet용 stem(7×7 stride 2 + maxpool)은
  - `conv1`을 **3×3 stride 1**로 바꾸고
  - `maxpool`을 제거합니다.
- 출제 포인트:
  - 모델 layer/구조의 의미와 동작 원리
- 빈칸 기준:
  - Conv2d layer
  - layer 제거/identity
  - Linear layer


In [ ]:
def build_resnet18_for_cifar10(num_classes: int = 10):
    model = models.resnet18(weights=None)  # ImageNet pretrained 없이 ResNet18 생성 (from scratch)
    print("\n" + "="*40)
    print("=== PyTorch model structure (Before) ===")  # 모델 구조 출력
    print("="*40 + "\n")
    print(model)  # 구조 출력

    model.conv1 = ???  # CIFAR용 stem
    model.maxpool = ???  # maxpool 제거
    model.fc = ???  # 최종 분류기 출력=10
    return model

model = build_resnet18_for_cifar10(num_classes).to(device)  # 모델 생성 후 device 이동

print("\n" + "="*40)
print("=== PyTorch model structure (After) ===")  # 모델 구조 출력
print("="*40 + "\n")
print(model)  # 구조 출력

num_params = sum(p.numel() for p in model.parameters())  # 파라미터 개수
print(f'params: {num_params:,}')  # 출력


#### 정답

??? 순서: `nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)`, `nn.Identity()`, `nn.Linear(model.fc.in_features, num_classes)`

#### 간략 해설

1. `nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)`: 이미지의 공간 패턴을 channel feature로 추출하는 핵심 CNN layer입니다.
2. `nn.Identity()`: CIFAR-10의 작은 32x32 입력에서는 초기 maxpool downsampling을 제거하기 위해 사용합니다.
3. `nn.Linear(model.fc.in_features, num_classes)`: 추출된 feature를 class 수에 맞는 logit으로 변환하는 분류 head입니다.


### 문제 3. 3) 학습 유틸리티
- 원본 파일/셀: `01_ResNet18_CIFAR10.ipynb` / code cell index `10`
- 목적: 목적:** 학습/평가 루프를 재사용 가능하게 만들어 실험을 단순화합니다.
- 관찰 포인트:
  - **관찰 포인트**
  - `model.train()` vs `model.eval()` 차이
  - AMP(Automatic Mixed Precision) 사용
  - 계산이 복잡한 곳은 FP32를 쓰고, 단순한 곳은 FP16을 섞어서 사용
- 출제 포인트:
  - 모델 결과와 정답을 이용한 학습
  - 평가 metric 및 예측 후처리
- 빈칸 기준:
  - 예측 class 선택
  - train mode
  - gradient 초기화
  - forward pass
  - loss 계산
  - eval mode
  - forward pass
  - loss 계산


In [ ]:
@dataclass
class TrainConfig:
    base_epochs_if_new: int = 20  # 처음 실험 시 epoch 수
    extra_epochs_if_resume: int = 10  # 체크포인트 존재 시 추가 실험 epoch 수
    lr: float = 0.1  # learning rate
    weight_decay: float = 5e-4  # L2 정규화
    momentum: float = 0.9  # SGD momentum
    label_smoothing: float = 0.0  # 필요 시 label smoothing, 예) [0, 1, 0] ->  [0.05, 0.9, 0.05]

cfg = TrainConfig()  # config 인스턴스 생성

def accuracy_top1(logits, targets):
    preds = ???  # 가장 큰 logit의 클래스 선택, logits = [B, C], preds = [B]
    return (preds == targets).float().mean().item()  # top-1 accuracy, scalar

scaler = torch.amp.GradScaler(enabled=torch.cuda.is_available())  # AMP용 scaler, 학습 속도를 높이고 메모리 사용량을 줄임
print(scaler)

def train_one_epoch(model, loader, optimizer, criterion):
    ???  # train mode(dropout/bn 동작 변경)
    total_loss, total_acc = 0.0, 0.0  # 누적 loss/acc
    n = 0  # 샘플 수 누적
    t0 = time.time()  # 시간 측정 시작
    for images, labels in loader:
        images = images.to(device, non_blocking=True)  # 입력을 GPU로 이동 (Asynchronous transfer)
        labels = labels.to(device, non_blocking=True)  # 라벨을 GPU로 이동 (Asynchronous transfer)

        ???  # gradient 초기화
        with torch.amp.autocast(device_type=device.type, enabled=torch.cuda.is_available()):  # AMP autocast
            logits = ???  # forward
            loss = ???  # loss 계산
        scaler.scale(loss).backward()  # scaled backward
        scaler.step(optimizer)  # optimizer step
        scaler.update()  # scaler 업데이트

        bs = images.size(0)  # batch size
        total_loss += loss.item() * bs  # batch loss 누적
        total_acc  += accuracy_top1(logits.detach(), labels) * bs  # logits값만 복사, batch acc 누적
        n += bs  # 샘플 수 누적

    dt = time.time() - t0  # epoch 수행 시간
    return total_loss / n, total_acc / n, dt  # 평균 loss/acc/시간

@torch.no_grad()   # Autograd off, 함수 내부에서 기울기 계산 멈춤
def evaluate(model, loader, criterion):
    ???  # eval mode, i.e., Dropout 비활성화, BN Running Stats 고정
    total_loss, total_acc = 0.0, 0.0  # 누적
    n = 0  # 샘플 수
    for images, labels in loader:
        images = images.to(device, non_blocking=True)  # GPU 이동
        labels = labels.to(device, non_blocking=True)  # GPU 이동
        logits = ???  # forward
        loss = ???  # loss
        bs = images.size(0)  # batch size
        total_loss += loss.item() * bs  # 누적
        total_acc  += accuracy_top1(logits, labels) * bs  # 누적
        n += bs  # 누적
    return total_loss / n, total_acc / n  # 평균 loss/acc

def plot_history(hist):
    epochs = np.arange(1, len(hist['train_loss'])+1)  # epoch index
    plt.figure(figsize=(12,4))
    plt.plot(epochs, hist['train_loss'], label='train_loss')  # train loss
    plt.plot(epochs, hist['val_loss'], label='val_loss')  # val loss
    plt.xlabel('epoch'); plt.ylabel('loss'); plt.legend(); plt.grid(True)
    plt.show()

    plt.figure(figsize=(12,4))
    plt.plot(epochs, hist['train_acc'], label='train_acc')  # train acc
    plt.plot(epochs, hist['val_acc'], label='val_acc')  # val acc
    plt.xlabel('epoch'); plt.ylabel('acc'); plt.legend(); plt.grid(True)
    plt.show()


#### 정답

??? 순서: `logits.argmax(dim=1)`, `model.train()`, `optimizer.zero_grad(set_to_none=True)`, `model(images)`, `criterion(logits, labels)`, `model.eval()`, `model(images)`, `criterion(logits, labels)`

#### 간략 해설

1. `logits.argmax(dim=1)`: 가장 큰 logit 위치가 모델이 선택한 class이므로 top-1 예측에 사용합니다.
2. `model.train()`: Dropout/BatchNorm 등을 학습 모드로 전환해 parameter update에 맞게 동작시킵니다.
3. `optimizer.zero_grad(set_to_none=True)`: PyTorch gradient는 누적되므로 새 step 전에 이전 gradient를 초기화합니다.
4. `model(images)`: forward pass에 해당하는 핵심 코드로, 이 위치의 입력/출력 흐름을 결정합니다.
5. `criterion(logits, labels)`: 모델 출력과 정답 label/mask를 비교해 학습 신호인 loss를 계산합니다.
6. `model.eval()`: 평가/추론에서는 Dropout을 끄고 BatchNorm running statistics를 사용해야 합니다.
7. `model(images)`: forward pass에 해당하는 핵심 코드로, 이 위치의 입력/출력 흐름을 결정합니다.
8. `criterion(logits, labels)`: 모델 출력과 정답 label/mask를 비교해 학습 신호인 loss를 계산합니다.


### 문제 4. 4) ResNet18 학습
- 원본 파일/셀: `01_ResNet18_CIFAR10.ipynb` / code cell index `12`
- 목적: 목적:** CIFAR-10에서 ResNet18을 학습하고, 가장 좋은 성능의 모델을 저장합니다.
- 관찰 포인트:
  - **관찰 포인트**
  - `resnet18_cifar10.pth`가 있으면 **그 모델을 로드**하고 **10 epoch만 추가 학습**합니다.
  - 파일이 없으면 **20 epoch 학습**합니다.
  - 체크포인트로 실험을 이어갈 수 있다는 점
- 출제 포인트:
  - 모델 결과와 정답을 이용한 학습
- 빈칸 기준:
  - loss 함수 설정
  - optimizer 설정


In [ ]:
criterion = ???  # 분류 loss
optimizer = ???  # Mini-Batch SGD optimizer with Momentum

ckpt_path = 'resnet18_cifar10.pth'  # best checkpoint 경로
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'lr': []}  # 기록용 dict

start_epoch = 0  # resume 시작 epoch
best_acc = -1.0  # best val acc

if os.path.exists(ckpt_path):  # 체크포인트가 있으면 resume
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)  # checkpoint 로드
    model.load_state_dict(ckpt['model'])  # 모델 가중치 로드
    start_epoch = int(ckpt.get('epoch', 0))  # 저장된 epoch
    best_acc = float(ckpt.get('val_acc', -1.0))  # 저장된 best acc
    epochs_to_run = cfg.extra_epochs_if_resume  # 추가 실험 epoch = 10
    print(f"Found checkpoint. Resume from epoch {start_epoch} (best val_acc={best_acc:.4f}). Train +{epochs_to_run} epochs.")
else:  # 체크포인트가 없으면 처음부터
    epochs_to_run = cfg.base_epochs_if_new  # 신규 실험 epoch = 20
    print(f"No checkpoint. Train {epochs_to_run} epochs from scratch.")

# 학습률을 T_max에 맞춰 코사인 곡선 형태로 변화
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs_to_run)  # 이번 실험 구간에 대해 cosine schedule

for e in range(1, epochs_to_run+1):
    epoch = start_epoch + e  # 실제 epoch 번호(누적)
    lr_now = optimizer.param_groups[0]['lr']  # 현재 lr, [ { 'params': [...], 'lr': 0.1, 'momentum': 0.9, 'weight_decay': 0.0005, ... } ]
    tr_loss, tr_acc, dt = train_one_epoch(model, train_loader, optimizer, criterion)  # 1 epoch train
    va_loss, va_acc = evaluate(model, test_loader, criterion)  # validation
    scheduler.step()  # lr 스케줄 업데이트

    history['train_loss'].append(tr_loss)  # 기록
    history['train_acc'].append(tr_acc)  # 기록
    history['val_loss'].append(va_loss)  # 기록
    history['val_acc'].append(va_acc)  # 기록
    history['lr'].append(lr_now)  # 기록

    print(f"Epoch {epoch:03d} | lr {lr_now:.4f} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | val loss {va_loss:.4f} acc {va_acc:.4f} | {dt:.1f}s")

    if va_acc > best_acc:  # best 갱신 시 저장
        best_acc = va_acc  # best 업데이트
        torch.save({'model': model.state_dict(), 'epoch': epoch, 'val_acc': va_acc, 'config': cfg.__dict__}, ckpt_path)  # 체크포인트 저장
        print(f"  Saved best checkpoint: {ckpt_path} (val_acc={va_acc:.4f})")

print('Best val acc:', best_acc)  # best 출력
plot_history(history)  # 실험 곡선

plt.figure(figsize=(12,3))
plt.plot(np.arange(1, epochs_to_run+1), history['lr'])  # lr 변화(이번 구간)
plt.xlabel('epoch'); plt.ylabel('lr'); plt.title('Learning rate schedule (this segment)'); plt.grid(True)
plt.show()


#### 정답

??? 순서: `nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)`, `torch.optim.SGD(model.parameters(), lr=cfg.lr, momentum=cfg.momentum, weight_decay=cfg.weight_decay)`

#### 간략 해설

1. `nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)`: 모델 출력과 정답 label/mask를 비교해 학습 신호인 loss를 계산합니다.
2. `torch.optim.SGD(model.parameters(), lr=cfg.lr, momentum=cfg.momentum, weight_decay=cfg.weight_decay)`: optimizer는 gradient를 어떤 규칙과 learning rate로 갱신할지 정합니다.


### 문제 5. 5) 추론(Inference)
- 원본 파일/셀: `01_ResNet18_CIFAR10.ipynb` / code cell index `14`
- 목적: 목적:** 학습된 모델이 실제로 어떤 클래스로 예측하는지 확인합니다.
- 관찰 포인트:
  - **관찰 포인트**
  - softmax 확률과 confidence(최대 확률)
- 출제 포인트:
  - 모델 결과와 정답을 이용한 학습
  - 평가 metric 및 예측 후처리
- 빈칸 기준:
  - eval mode
  - forward pass
  - 확률 변환


In [ ]:
ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)  # best checkpoint 로드
model.load_state_dict(ckpt['model'])  # 최신 모델 가중치 적용
???  # eval 모드, i.e., Dropout 비활성화, BN Running Stats 고정
print('Loaded checkpoint epoch:', ckpt.get('epoch'), 'val_acc:', ckpt.get('val_acc'))  # 정보 출력

@torch.no_grad()  # 자동 미분(Autograd) 기능 비활성화
def predict_batch(model, images):
    logits = ???  # raw logits
    probs = ???  # 확률로 변환
    conf, pred = probs.max(dim=1)  # 최대 확률(conf)과 클래스(pred)
    return pred, conf, probs

images, labels = next(iter(test_loader))  # test 배치 하나
images = images.to(device)  # GPU 이동
labels = labels.to(device)  # GPU 이동
pred, conf, probs = predict_batch(model, images)  # 예측

k = 16  # 시각화 샘플 수
imgs_dn = denorm(images[:k].cpu()).clamp(0,1)  # 역정규화

grid = torchvision.utils.make_grid(imgs_dn, nrow=8)  # 그리드
plt.figure(figsize=(12,6))
plt.imshow(grid.permute(1,2,0))  # HWC로 변환
plt.axis('off')
plt.title('Inference samples')
plt.show()

for i in range(k):
    gt = class_names[int(labels[i])]  # 정답 클래스명
    pr = class_names[int(pred[i])]  # 예측 클래스명
    cf = float(conf[i])  # confidence
    print(f"[{i:02d}] GT={gt:>10s} | Pred={pr:>10s} | conf={cf:.3f}")  # 출력


#### 정답

??? 순서: `model.eval()`, `model(images)`, `F.softmax(logits, dim=1)`

#### 간략 해설

1. `model.eval()`: 평가/추론에서는 Dropout을 끄고 BatchNorm running statistics를 사용해야 합니다.
2. `model(images)`: forward pass에 해당하는 핵심 코드로, 이 위치의 입력/출력 흐름을 결정합니다.
3. `F.softmax(logits, dim=1)`: class logit을 class별 확률 분포로 바꿔 confidence나 attention 확률을 해석합니다.


### 문제 6. 6) 성능 분석: Confusion Matrix / 클래스별 정확도 / 오분류 샘플
- 원본 파일/셀: `01_ResNet18_CIFAR10.ipynb` / code cell index `16`
- 목적: 목적:** 단순 overall accuracy 말고, **어떤 클래스가 서로 헷갈리는지**를 분석합니다.
- 관찰 포인트:
  - **관찰 포인트**
  - Confusion Matrix에서 off-diagonal가 큰 위치 = 헷갈리는 클래스 쌍
  - 클래스별 accuracy로 데이터/모델의 약점 찾기
- 출제 포인트:
  - 평가 metric 및 예측 후처리 원리
- 빈칸 기준:
  - 평가/추론에서 gradient 비활성화
  - 평가 모드 전환
  - confusion matrix에서 정답/예측 쌍 누적
  - 클래스별 accuracy 계산


In [ ]:
???
def collect_predictions(model, loader):
    ???  # eval 모드
    all_preds, all_labels, all_images = [], [], []  # 누적 리스트
    for images, labels in loader:
        images = images.to(device)  # GPU 이동
        labels = labels.to(device)  # GPU 이동
        preds, conf, _ = predict_batch(model, images)  # 예측
        all_preds.append(preds.cpu())  # CPU로 모아두기
        all_labels.append(labels.cpu())  # CPU로 모아두기
        all_images.append(images.cpu())  # 이미지도 저장(오분류 시각화용)
    return torch.cat(all_images), torch.cat(all_preds), torch.cat(all_labels)  # 전체 텐서로 결합

def confusion_and_perclass(preds, labels, num_classes):
    cm = torch.zeros((num_classes, num_classes), dtype=torch.int64)  # confusion matrix 초기화
    for t, p in zip(labels, preds):
        ???  # GT=t, Pred=p 카운트 증가
    per_class_acc = ???  # 클래스별 정확도
    return cm, per_class_acc

images_all, preds_all, labels_all = collect_predictions(model, test_loader)  # test 전체 예측 수집
cm, pacc = confusion_and_perclass(preds_all, labels_all, num_classes)  # confusion + per-class

plt.figure(figsize=(8,7))
plt.imshow(cm.numpy())  # confusion matrix 이미지화
plt.title('Confusion Matrix (Test)')
plt.xlabel('Pred'); plt.ylabel('GT')
plt.xticks(range(num_classes), class_names, rotation=45, ha='right')
plt.yticks(range(num_classes), class_names)
plt.colorbar()
plt.tight_layout()
plt.show()

print('Per-class accuracy:')
for i, a in enumerate(pacc.tolist()):
    print(f"{class_names[i]:>10s}: {a:.3f}")  # 클래스별 정확도 출력

# 오분류 샘플 랜덤 16개 표시
mis_idx = (preds_all != labels_all).nonzero(as_tuple=False).squeeze(1)  # 오분류 인덱스, squeeze(1) : 1번째 인덱스 차원 제거
print('num misclassified:', int(mis_idx.numel()))  # 오분류 개수
if mis_idx.numel() > 0:
    sel = mis_idx[torch.randperm(mis_idx.numel())[:16]]  # 랜덤 16개 선택
    imgs = denorm(images_all[sel]).clamp(0,1)  # 역정규화
    grid = torchvision.utils.make_grid(imgs, nrow=8)  # 그리드
    plt.figure(figsize=(12,5))
    plt.imshow(grid.permute(1,2,0))  # 표시
    plt.axis('off')
    plt.title('Random misclassified examples')
    plt.show()

    for j, idx in enumerate(sel.tolist()):
        gt = class_names[int(labels_all[idx])]  # GT
        pr = class_names[int(preds_all[idx])]  # Pred
        print(f"[{j:02d}] idx={idx:5d} GT={gt:>10s} Pred={pr:>10s}")  # 출력


#### 정답

??? 순서: `@torch.no_grad()`, `model.eval()`, `cm[int(t), int(p)] += 1`, `cm.diagonal().float() / torch.clamp(cm.sum(dim=1).float(), min=1.0)`

#### 간략 해설

1. `@torch.no_grad()`: 평가에서는 gradient를 저장할 필요가 없으므로 메모리와 연산을 줄입니다.
2. `model.eval()`: Dropout/BatchNorm을 평가 모드로 고정해 일관된 예측을 얻습니다.
3. `cm[int(t), int(p)] += 1`: confusion matrix는 행을 정답, 열을 예측으로 두고 각 정답/예측 쌍의 빈도를 누적합니다.
4. `cm.diagonal().float() / torch.clamp(cm.sum(dim=1).float(), min=1.0)`: 대각선은 클래스별 정답 수이고 행 합은 해당 정답 클래스의 전체 수이므로 클래스별 accuracy가 됩니다.


## 02_ViT_CIFAR10.ipynb

### 문제 7. 1) 데이터 로딩 & 전처리
- 원본 파일/셀: `02_ViT_CIFAR10.ipynb` / code cell index `5`
- 목적: 목적:** CIFAR-10(컬러 32×32)를 로드하고, 모델 입력 텐서 형태를 확인합니다.
- 관찰 포인트:
  - **관찰 포인트**
  - 입력 텐서 shape: `B×C×H×W` (CIFAR-10는 `C=3`)
  - 학습/테스트 로더 구성과 배치 단위 학습의 의미
- 출제 포인트:
  - 학습/평가 의도에 맞는 데이터 구성과 이미지 변형
- 빈칸 기준:
  - transform 순차 구성
  - random crop augmentation
  - horizontal flip augmentation
  - tensor 변환
  - 채널 정규화
  - transform 순차 구성
  - tensor 변환
  - 채널 정규화


In [ ]:
# CIFAR-10에서 널리 쓰는 normalize 값
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)  # 채널별 평균(R,G,B)
CIFAR10_STD  = (0.2023, 0.1994, 0.2010)  # 채널별 표준편차(R,G,B)

train_tfms = ???([  # train transform(augmentation 포함)
    ???,      # 32x32를 padding 후 random crop
    ???,         # 좌우 반전
    ???,                     # PIL -> torch tensor (C,H,W)
    ???,  # 정규화
])

test_tfms = ???([  # test transform(augmentation 없음)
    ???,                     # 텐서 변환
    ???,  # 정규화
])

data_root = './data'  # 데이터 저장 경로
train_set = torchvision.datasets.CIFAR10(root=data_root, train=True, download=True, transform=train_tfms)  # train set
test_set  = torchvision.datasets.CIFAR10(root=data_root, train=False, download=True, transform=test_tfms)  # test set

class_names = train_set.classes  # 클래스 이름 목록
num_classes = len(class_names)   # 클래스 개수(10)
print('classes:', class_names)   # 클래스 출력

# 이미지 사이즈 출력
x0, y0 = train_set[0]  # 한 샘플 로드(이미 transform 적용된 텐서)
print('Sample image tensor shape (C,H,W):', tuple(x0.shape))  # (3,32,32)

batch_size = 256  # 배치 크기(A10이면 보통 여유)
num_workers = min(8, os.cpu_count() or 2)  # dataloader worker 수(환경에 맞게)

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)  # train loader
test_loader  = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)  # test loader

print('Total batch size (train: %d, test: %d)' % (len(train_loader), len(test_loader)))  # 배치 개수 확인


#### 정답

??? 순서: `T.Compose`, `T.RandomCrop(32, padding=4)`, `T.RandomHorizontalFlip()`, `T.ToTensor()`, `T.Normalize(CIFAR10_MEAN, CIFAR10_STD)`, `T.Compose`, `T.ToTensor()`, `T.Normalize(CIFAR10_MEAN, CIFAR10_STD)`

#### 간략 해설

1. `T.Compose`: 여러 transform을 지정한 순서대로 적용해야 하므로 사용합니다.
2. `T.RandomCrop(32, padding=4)`: 학습 이미지를 무작위로 자르고 padding을 주어 위치 변화에 덜 민감하게 만듭니다.
3. `T.RandomHorizontalFlip()`: 좌우 반전을 추가해 학습 데이터 다양성을 늘리는 augmentation입니다.
4. `T.ToTensor()`: PIL 이미지를 PyTorch가 처리하는 C,H,W tensor로 바꿉니다.
5. `T.Normalize(CIFAR10_MEAN, CIFAR10_STD)`: 채널별 평균과 표준편차로 입력 분포를 맞춰 학습을 안정화합니다.
6. `T.Compose`: 여러 transform을 지정한 순서대로 적용해야 하므로 사용합니다.
7. `T.ToTensor()`: PIL 이미지를 PyTorch가 처리하는 C,H,W tensor로 바꿉니다.
8. `T.Normalize(CIFAR10_MEAN, CIFAR10_STD)`: 채널별 평균과 표준편차로 입력 분포를 맞춰 학습을 안정화합니다.


### 문제 8. 2) ViT 모델 구성 개요
- 원본 파일/셀: `02_ViT_CIFAR10.ipynb` / code cell index `8`
- 출제 포인트:
  - 모델 layer/구조의 의미와 동작 원리
- 빈칸 기준:
  - LayerNorm
  - Linear layer
  - ReLU activation
  - Linear layer
  - Linear layer
  - Linear layer
  - layer 제거/identity
  - patch rearrange


In [ ]:
def pair(t):  # image_size/patch_size를 (H,W) 형태로 통일하는 유틸
    return t if isinstance(t, tuple) else (t, t)


class PreNorm(nn.Module):  # Transformer의 Pre-LN 구조: LayerNorm 후 블록 실행
    def __init__(self, dim, fn):
        super().__init__()
        self.norm = ???  # 토큰 임베딩 차원(dim) 기준 LayerNorm -> 패치별 정규화
        self.fn = fn
    def forward(self, x, **kwargs):
        return self.fn(self.norm(x), **kwargs)  # 정규화된 토큰을 Attention/FFN에 전달

class FeedForward(nn.Module):  # Transformer의 FFN(MLP) 블록 -> 패치 자체의 정보를 더 깊게 분석
    def __init__(self, dim, hidden_dim, dropout = 0.):
        super().__init__()
        self.net = nn.Sequential(  # FFN: Linear→GELU→Dropout→Linear→Dropout 구성
            ???,
            ???,
            nn.Dropout(dropout),
            ???,
            nn.Dropout(dropout)
        )
    def forward(self, x):
        return self.net(x)

class Attention(nn.Module):  # 멀티헤드 Self-Attention(MHSA) 구현
    def __init__(self, dim, heads = 4, dim_head = 64, dropout = 0.):
        super().__init__()
        inner_dim = dim_head *  heads  # 전체 헤드 차원 = head 수 × head 차원
        project_out = not (heads == 1 and dim_head == dim)

        self.heads = heads  # 멀티헤드 개수 저장
        self.scale = dim_head ** -0.5  # Scaled dot-product를 위한 스케일(1/√d)

        self.attend = nn.Softmax(dim = -1)  # attention score를 확률로 변환(softmax), 마지막 차원 : sequence length N
        self.last_attn = None  # (학습/추론 시) 마지막 forward에서의 attention map 저장용(시각화/분석)  # attention score를 확률로 변환(softmax), 마지막 차원 : sequence length N
        self.to_qkv = ???  # 입력 토큰→Q,K,V를 한 번에 선형 변환

        self.to_out = nn.Sequential(  # 헤드들을 합친 뒤 출력 투영 + 드롭아웃
            ???,
            nn.Dropout(dropout)
        ) if project_out else ???

    def forward(self, x):
        b, n, _, h = *x.shape, self.heads   # x = [batch Size, tokens, embedding dimension]
        qkv = self.to_qkv(x).chunk(3, dim = -1)  # 선형변환 결과를 Q,K,V로 분할 -> (Q, K ,V)
        q, k, v = map(lambda t: ??? -> b h n d', h = h), qkv)  # multi-head 연산을 위해 각 head 기준 n x d 로 분리

        dots = einsum('b h i d, b h j d -> b h i j', q, k) * self.scale  # 각 토큰 간 유사도(Q·Kᵀ) 계산 후 스케일 적용

        attn = self.attend(dots)  # 토큰 간 가중치(attention map) 생성
        self.last_attn = attn.detach()  # 그래프에서 분리(detach)해서 저장 (시각화 목적)

        out = einsum('b h i j, b h j d -> b h i d', attn, v)  # attention 가중합으로 새로운 토큰 표현(out) 계산
        out = rearrange(out, 'b h n d -> b n (h d)')  # 헤드 차원을 다시 합쳐 (batch, tokens, dim)로 복원
        return self.to_out(out)  # 최종 투영을 거쳐 Attention 블록 출력 반환

class Transformer(nn.Module):  # Encoder block을 여러 층(depth) 쌓는 Transformer
    def __init__(self, dim, depth, heads, dim_head, mlp_dim, dropout = 0.):
        super().__init__()
        self.layers = nn.ModuleList([])  # 각 층(Attention+FFN)을 담을 컨테이너
        for _ in range(depth):  # depth 만큼 Encoder block 반복 생성
            self.layers.append(nn.ModuleList([  # 한 층 = (PreNorm+Attention) + (PreNorm+FFN)
                PreNorm(dim, Attention(dim, heads = heads, dim_head = dim_head, dropout = dropout)),
                PreNorm(dim, FeedForward(dim, mlp_dim, dropout = dropout))
            ]))
    def forward(self, x):
        for attn, ff in self.layers:
            x = attn(x) + x  # Residual 연결: Attention 결과를 입력에 더해 정보 보존
            x = ff(x) + x  # Residual 연결: FFN 결과를 입력에 더해 정보 보존
        return x


#### 정답

??? 순서: `nn.LayerNorm(dim)`, `nn.Linear(dim, hidden_dim)`, `nn.ReLU()`, `nn.Linear(hidden_dim, dim)`, `nn.Linear(dim, inner_dim * 3, bias = False)`, `nn.Linear(inner_dim, dim)`, `nn.Identity()`, `rearrange(t, 'b n (h d)`

#### 간략 해설

1. `nn.LayerNorm(dim)`: Transformer 계열에서 token feature를 정규화해 attention/MLP 학습을 안정화합니다.
2. `nn.Linear(dim, hidden_dim)`: 추출된 feature를 class 수에 맞는 logit으로 변환하는 분류 head입니다.
3. `nn.ReLU()`: ReLU activation에 해당하는 핵심 코드로, 이 위치의 입력/출력 흐름을 결정합니다.
4. `nn.Linear(hidden_dim, dim)`: 추출된 feature를 class 수에 맞는 logit으로 변환하는 분류 head입니다.
5. `nn.Linear(dim, inner_dim * 3, bias = False)`: 추출된 feature를 class 수에 맞는 logit으로 변환하는 분류 head입니다.
6. `nn.Linear(inner_dim, dim)`: 추출된 feature를 class 수에 맞는 logit으로 변환하는 분류 head입니다.
7. `nn.Identity()`: CIFAR-10의 작은 32x32 입력에서는 초기 maxpool downsampling을 제거하기 위해 사용합니다.
8. `rearrange(t, 'b n (h d)`: 이미지를 patch 단위 token sequence로 바꾸기 위한 shape 변환입니다.


### 문제 9. 2-1) ViTConfig + ViT(모델 본체) 구현
- 원본 파일/셀: `02_ViT_CIFAR10.ipynb` / code cell index `10`
- 목적: 목적:** ViT의 핵심 구성 요소(패치 임베딩, CLS 토큰, 위치 임베딩, Transformer Encoder 블록, 분류 헤드)를 **코드로 직접 확인**합니다.
- 관찰 포인트:
  - **관찰 포인트**
  - `image_size`, `patch_size` → 패치 개수(`num_patches`)와 토큰 시퀀스 길이가 어떻게 결정되는지
  - `depth`(블록 개수), `heads`(멀티헤드 수), `dim`(토큰 임베딩 차원)이 **연산량/표현력**에 어떤 영향을 주는지
  - **Pre-LN + Residual** 구조가 어디에 적용되는지(안정적인 학습을 위한 표준 패턴)
- 출제 포인트:
  - 모델 layer/구조의 의미와 동작 원리
- 빈칸 기준:
  - Linear layer
  - LayerNorm
  - Linear layer


In [ ]:
@dataclass
class ViTConfig:
    # 입력/토큰화 관련
    image_size: 'Union[int, Tuple[int, int]]'          # 입력 이미지 크기 (H, W) 또는 정수(정수면 정사각형)
    patch_size: 'Union[int, Tuple[int, int]]'          # 패치 크기 (Ph, Pw) 또는 정수
    channels: int = 3                          # 입력 채널 수 (CIFAR-10=3, MNIST=1)

    # 모델 본체(Encoder) 관련
    dim: int = 64                              # 토큰 임베딩 차원(Transformer의 hidden size)
    depth: int = 6                             # Encoder 블록 개수(= Transformer layer 수)
    heads: int = 4                             # Multi-Head Self-Attention의 head 개수
    dim_head: int = 64                         # 각 head의 Q/K/V 차원
    mlp_dim: int = 128                         # FFN(MLP) 중간 차원

    # 분류/출력 관련
    num_classes: int = 10                      # 분류 클래스 수
    pool: str = "cls"                          # "cls": CLS 토큰 사용, "mean": 토큰 평균 풀링
    dropout: float = 0.0                       # Encoder 내부 dropout(Attention/FFN)
    emb_dropout: float = 0.0                   # 패치+포지션 임베딩 단계 dropout


class ViT(nn.Module):
    """Vision Transformer (ViT)
    - 이미지를 패치로 쪼개 토큰 시퀀스를 만들고
    - CLS 토큰 + 위치 임베딩을 더한 뒤
    - Transformer Encoder로 전역(Self-Attention) 관계를 학습하여
    - CLS(또는 mean pool) 표현으로 분류합니다.
    """

    def __init__(self, cfg: ViTConfig):
        super().__init__()

        # 1) 입력 해상도/패치 크기를 (H,W) 튜플로 정규화
        image_height, image_width = pair(cfg.image_size)
        patch_height, patch_width = pair(cfg.patch_size)

        # 2) 패치가 이미지에 딱 나누어 떨어져야 (h, w) 그리드가 정확히 형성됨
        assert image_height % patch_height == 0 and image_width % patch_width == 0, "Image dimensions must be divisible by the patch size."

        # 3) 패치 개수(=토큰 개수)와 한 패치의 펼친 차원 계산
        num_patches = (image_height // patch_height) * (image_width // patch_width)
        patch_dim = cfg.channels * patch_height * patch_width

        # 4) 풀링 방식 검증: CLS 토큰을 쓸지 mean pool을 쓸지 선택
        assert cfg.pool in {"cls", "mean"}, "pool must be 'cls' or 'mean'"

        # 5) 패치 토큰화: (B,C,H,W) -> (B, N, patch_dim) -> (B, N, dim)
        self.to_patch_embedding = nn.Sequential(
            # 패치 그리드로 자른 뒤, 각 패치를 1D 벡터로 펼쳐 토큰 시퀀스를 만듦
            Rearrange(
                "b c (h p1) (w p2) -> b (h w) (p1 p2 c)",
                p1=patch_height,
                p2=patch_width,
            ),
            # 펼친 패치 벡터를 Transformer hidden size(dim)로 선형 투영
            ???,
        )

        # 6) CLS 토큰 + 위치 임베딩(학습 파라미터)
        self.cls_token = nn.Parameter(torch.randn(1, 1, cfg.dim))
        self.pos_embedding = nn.Parameter(torch.randn(1, num_patches + 1, cfg.dim))
        self.dropout = nn.Dropout(cfg.emb_dropout)

        # 7) Transformer Encoder 스택(Attention + FFN + Residual/PreNorm)
        self.transformer = Transformer(
            dim=cfg.dim,
            depth=cfg.depth,
            heads=cfg.heads,
            dim_head=cfg.dim_head,
            mlp_dim=cfg.mlp_dim,
            dropout=cfg.dropout,
        )

        self.pool = cfg.pool

        # 8) 분류 헤드: (CLS/mean) 표현 -> LayerNorm -> Linear(logits)
        self.mlp_head = nn.Sequential(
            ???,
            ???,
        )

    def forward(self, img):
        # A) 패치 임베딩으로 토큰 시퀀스 생성: (B,C,H,W) -> (B,N,dim)
        x = self.to_patch_embedding(img)
        b, n, _ = x.shape

        # B) CLS 토큰을 배치만큼 복제해 시퀀스 맨 앞에 붙임: (B,1,dim)
        cls_tokens = repeat(self.cls_token, "1 1 d -> b 1 d", b=b)
        x = torch.cat((cls_tokens, x), dim=1)  # (B, N+1, dim)

        # C) 위치 임베딩을 더해 토큰 순서(공간 위치) 정보를 주입
        x = x + self.pos_embedding[:, : (n + 1)]   # (B, N+1, dim) + (1, N+1, dim) -> Broadcasting
        x = self.dropout(x)

        # D) Encoder를 통과하며 전역 의존성(Self-Attention) 학습
        x = self.transformer(x)  # (B, N+1, dim)

        # E) 이미지 표현 벡터 선택: CLS 토큰(0번) 또는 mean pooling
        x = x[:, 0] if self.pool == "cls" else x.mean(dim=1)   # (B, dim)

        # F) 최종 logits 출력 (softmax는 CrossEntropyLoss 내부에서 처리)
        return self.mlp_head(x)


#### 정답

??? 순서: `nn.Linear(patch_dim, cfg.dim)`, `nn.LayerNorm(cfg.dim)`, `nn.Linear(cfg.dim, cfg.num_classes)`

#### 간략 해설

1. `nn.Linear(patch_dim, cfg.dim)`: 추출된 feature를 class 수에 맞는 logit으로 변환하는 분류 head입니다.
2. `nn.LayerNorm(cfg.dim)`: Transformer 계열에서 token feature를 정규화해 attention/MLP 학습을 안정화합니다.
3. `nn.Linear(cfg.dim, cfg.num_classes)`: 추출된 feature를 class 수에 맞는 logit으로 변환하는 분류 head입니다.


### 문제 10. 2-2) ViT 하이퍼파라미터 설정 (vit_cfg)
- 원본 파일/셀: `02_ViT_CIFAR10.ipynb` / code cell index `12`
- 목적: 목적:** ViT의 구조를 결정하는 설정값을 한 곳에서 정의하고, 아래 `ViT(vit_cfg)`가 **어떤 형태의 모델**을 만드는지 연결해 봅니다.
- 관찰 포인트:
  - **관찰 포인트**
  - `patch_size`가 작아질수록 토큰 수가 늘어 **Self-Attention 비용**이 증가합니다.
  - `depth/heads/dim`은 성능뿐 아니라 **학습 시간·메모리**에도 직접 영향을 줍니다.
  - CIFAR-10(32×32, RGB)에서도 과도한 설정은 **학습 불안정/과적합**을 유발할 수 있습니다.
- 출제 포인트:
  - 모델 결과와 정답을 이용한 학습
- 빈칸 기준:
  - optimizer 설정


In [ ]:
# ViT 설정을 한 곳에 모아두면(=config) 모델 구조를 한눈에 파악하기 쉽습니다.
vit_cfg = ViTConfig(
        image_size=32,      # CIFAR-10: 32x32
    patch_size=4,       # 4x4 패치 -> (32/4)^2 = 64개 패치 토큰
        channels=3,         # CIFAR-10은 컬러(3채널)
    dim=64,             # 토큰 임베딩 차원
    depth=6,            # Encoder 블록 수
    heads=4,            # MHSA head 수
    dim_head=64,        # head당 Q/K/V 차원
    mlp_dim=128,        # FFN 중간 차원
    num_classes=10,     # 0~9
    pool="cls",         # CLS 토큰으로 분류
    dropout=0.0,        # Encoder dropout
    emb_dropout=0.0,    # 임베딩 dropout
)

model = ViT(vit_cfg).to(device)
optimizer = ???    # Momentum + RMSProp


#### 정답

??? 순서: `optim.Adam(model.parameters(), lr=0.003)`

#### 간략 해설

1. `optim.Adam(model.parameters(), lr=0.003)`: optimizer는 gradient를 어떤 규칙과 learning rate로 갱신할지 정합니다.


### 문제 11. 4) Training (학습 루프)
- 원본 파일/셀: `02_ViT_CIFAR10.ipynb` / code cell index `18`
- 목적: 목적:** ViT를 CIFAR-10 학습 데이터로 학습시켜, *토큰 기반 분류가 실제로 동작*하는 것을 확인합니다.
- 관찰 포인트:
  - **관찰 포인트:**
  - `model.train()` 상태에서 **드롭아웃/정규화**가 어떻게 동작하는지
  - 손실(loss)이 줄어드는지(학습이 되는지)
- 출제 포인트:
  - 모델 결과와 정답을 이용한 학습
- 빈칸 기준:
  - train mode
  - gradient 초기화
  - forward pass
  - loss 계산
  - backpropagation
  - parameter update


In [ ]:
def train_epoch(model, optimizer, data_loader, loss_history):
    """한 epoch 학습을 수행하고, 진행 상황을 epoch당 '딱 5번'만 출력합니다."""
    total_samples = len(data_loader.dataset)
    ???

    # epoch당 출력 5회(0%, 25%, 50%, 75%, 마지막)로 고정
    n_steps = len(data_loader)
    log_steps = {
        0,
        max(0, n_steps // 4),
        max(0, n_steps // 2),
        max(0, (3 * n_steps) // 4),
        n_steps - 1,
    }

    for i, (data, target) in enumerate(data_loader):
        data = data.to(device)
        target = target.to(device)

        ???
        output = ???  # Softmax
        loss = ???           # Negative Log Likelihood
        # nn.CrossEntropyLoss(model(data), target)
        ???
        ???

        if i in log_steps:
            seen = min((i + 1) * len(data), total_samples)
            print(f"[{seen:5d}/{total_samples:5d} ({100.0 * (i+1) / n_steps:5.1f}%)]  Loss: {loss.item():.4f}")
            loss_history.append(loss.item())

#### 정답

??? 순서: `model.train()`, `optimizer.zero_grad()`, `F.log_softmax(model(data), dim=1)`, `F.nll_loss(output, target)`, `loss.backward()`, `optimizer.step()`

#### 간략 해설

1. `model.train()`: Dropout/BatchNorm 등을 학습 모드로 전환해 parameter update에 맞게 동작시킵니다.
2. `optimizer.zero_grad()`: PyTorch gradient는 누적되므로 새 step 전에 이전 gradient를 초기화합니다.
3. `F.log_softmax(model(data), dim=1)`: class logit을 class별 확률 분포로 바꿔 confidence나 attention 확률을 해석합니다.
4. `F.nll_loss(output, target)`: 모델 출력과 정답 label/mask를 비교해 학습 신호인 loss를 계산합니다.
5. `loss.backward()`: loss를 기준으로 각 parameter의 gradient를 계산합니다.
6. `optimizer.step()`: 계산된 gradient를 이용해 모델 parameter를 갱신합니다.


### 문제 12. 4) Training (학습 루프)
- 원본 파일/셀: `02_ViT_CIFAR10.ipynb` / code cell index `19`
- 목적: 목적:** ViT를 CIFAR-10 학습 데이터로 학습시켜, *토큰 기반 분류가 실제로 동작*하는 것을 확인합니다.
- 관찰 포인트:
  - **관찰 포인트:**
  - `model.train()` 상태에서 **드롭아웃/정규화**가 어떻게 동작하는지
  - 손실(loss)이 줄어드는지(학습이 되는지)
- 출제 포인트:
  - 모델 결과와 정답을 이용한 학습
- 빈칸 기준:
  - eval mode
  - forward pass
  - loss 계산


In [ ]:
def evaluate(model, data_loader, loss_history):
    """test_loader로 1회 평가하고 (loss, acc)를 반환합니다."""
    ???

    total_samples = len(data_loader.dataset)
    correct_samples = 0
    total_loss = 0.0

    with torch.no_grad():
        for data, target in data_loader:
            data = data.to(device)
            target = target.to(device)

            output = ???
            loss = ???
            _, pred = torch.max(output, dim=1)

            total_loss += float(loss.item())
            correct_samples += int(pred.eq(target).sum().item())

    avg_loss = total_loss / total_samples
    acc = correct_samples / total_samples

    loss_history.append(avg_loss)
    print(
        f"Average test loss: {avg_loss:.4f}  "
        f"Accuracy: {correct_samples:5d}/{total_samples:5d} ({acc*100:.2f}%)\n"
    )
    return avg_loss, acc


#### 정답

??? 순서: `model.eval()`, `F.log_softmax(model(data), dim=1)`, `F.nll_loss(output, target, reduction='sum')`

#### 간략 해설

1. `model.eval()`: 평가/추론에서는 Dropout을 끄고 BatchNorm running statistics를 사용해야 합니다.
2. `F.log_softmax(model(data), dim=1)`: class logit을 class별 확률 분포로 바꿔 confidence나 attention 확률을 해석합니다.
3. `F.nll_loss(output, target, reduction='sum')`: 모델 출력과 정답 label/mask를 비교해 학습 신호인 loss를 계산합니다.


### 문제 13. 6) Testing (Attention)
- 원본 파일/셀: `02_ViT_CIFAR10.ipynb` / code cell index `24`
- 목적: 목적:** ViT의 예측이 어떤 **패치(토큰)** 정보에 의존하는지, CLS 토큰의 attention을 통해 직관적으로 확인합니다.
- 관찰 포인트:
  - **관찰 포인트:**
  - CLS 토큰이 **어떤 위치(패치)** 에 강하게 attention을 주는지
  - layer/head를 바꾸면 attention 패턴이 어떻게 달라지는지
  - (선택) **정답/오답 샘플**에서 attention이 얼마나 설득력 있게 보이는지
- 출제 포인트:
  - 모델 결과와 정답을 이용한 학습
  - 평가 metric 및 예측 후처리
- 빈칸 기준:
  - eval mode
  - eval mode
  - forward pass
  - 예측 class 선택
  - eval mode
  - forward pass
  - 예측 class 선택


In [ ]:
# (실습) Testing (Attention): ViT가 '어디를 보고' 예측했는지(=CLS→Patch Attention) 시각화
# - 마지막 logits는 CLS 토큰에서 나오므로, "CLS 토큰이 어떤 패치 토큰들을 참고했는지"를 보면 모델의 단서를 얻을 수 있습니다.
# - 실무/연구 관점에서도 초기 레이어(layer_idx=0)는 저수준/분산된 패턴이 많아 해석이 애매한 경우가 많아서,
#   기본값은 "맨 마지막 Transformer 블록"의 attention을 보는 쪽이 직관적입니다.
# - 또한 'head 평균'뿐 아니라 '각 head'가 서로 다른 패턴을 볼 수 있으므로, 여러 head를 한 번에 오버레이해서 비교합니다.

import matplotlib.pyplot as plt
import torch.nn.functional as F
import numpy as np
import random


def denormalize_cifar10_chw(img_chw: torch.Tensor):
    """정규화된 CIFAR-10 이미지(C,H,W)를 시각화용(0~1)으로 복원합니다."""
    mean = torch.tensor(CIFAR10_MEAN, device=img_chw.device)[:, None, None]
    std  = torch.tensor(CIFAR10_STD,  device=img_chw.device)[:, None, None]
    x = img_chw * std + mean
    return x.clamp(0, 1)


def chw_to_hwc_np(img_chw: torch.Tensor) -> np.ndarray:
    """(C,H,W) 텐서를 matplotlib imshow용 (H,W,C) numpy로 변환합니다.
    - CIFAR-10은 정규화가 적용되어 있으므로 denormalize 후 표시합니다.
    """
    x = denormalize_cifar10_chw(img_chw)  # (C,H,W), 0~1
    x = x.detach().cpu()
    # (C,H,W) -> (H,W,C)
    x_hwc = x.permute(1, 2, 0).numpy()
    return x_hwc

def imshow_chw(ax, img_chw: torch.Tensor, title: str = None):
    """CHW 텐서를 안전하게 imshow합니다."""
    x_hwc = chw_to_hwc_np(img_chw)
    ax.imshow(x_hwc)
    ax.axis("off")
    if title is not None:
        ax.set_title(title)


def _extract_last_attn(model, layer_idx: int):
    """지정한 encoder 블록(layer)의 Attention 모듈이 저장해둔 last_attn을 꺼냅니다.
    - Attention.forward에서 self.last_attn = attn.detach()로 저장하도록 구현되어 있어야 합니다.
    - 반환 shape: [B, H, N, N] (B=batch, H=head, N=tokens=1+num_patches)
    """
    # transformer.layers[layer_idx] = (PreNorm(Attention), PreNorm(FFN)) 형태를 가정
    attn_module = model.transformer.layers[layer_idx][0].fn  # [0]:Attention, [1]: FF, PreNorm(fn=Attention)
    attn = attn_module.last_attn
    if attn is None:
        raise RuntimeError("Attention.last_attn이 비어 있습니다. 먼저 model(x)를 한 번 forward 해야 합니다.")
    return attn  # [B,H,N,N]

def _cls_to_patch_map(attn_bhNN, head_idx=None):
    """[B,H,N,N] attention에서 CLS→Patch attention만 뽑아 1D로 반환합니다.
    - CLS 토큰 index = 0
    - Patch 토큰 index = 1..N
    - head_idx=None이면 head 평균을 사용합니다.
    반환: cls_attn_1d shape = [num_patches]
    """
    # CLS(0) 가 각 토큰(j)에 주는 attention: attn[..., 0, j]
    if head_idx is None:
        # head 평균: [B,N] (여기서는 B=1 가정)
        cls_attn = attn_bhNN.mean(dim=1)[0, 0, 1:]  # 평균 관심지도 [1, H, N+1, N+1] → [1, N+1, N+1] → [N]
    else:
        cls_attn = attn_bhNN[0, head_idx, 0, 1:]    # 특정헤드 관심지도 [N]
    return cls_attn

def _to_2d_grid(cls_attn_1d, grid_h: int, grid_w: int):
    """[num_patches] 1D attention을 (grid_h, grid_w)로 reshape합니다."""
    attn_2d = cls_attn_1d.reshape(grid_h, grid_w)
    # 0~1 범위로 정규화(시각화 안정성)
    attn_2d = attn_2d - attn_2d.min()
    attn_2d = attn_2d / (attn_2d.max() + 1e-8)
    return attn_2d

def _upsample_to_image(attn_2d, out_h: int, out_w: int):
    """(grid_h,grid_w) attention을 (out_h,out_w)로 bilinear upsample합니다."""
        # attn_2d가 이미 Tensor일 수도 있으므로, 불필요한 tensor(...) 재구성을 피합니다.
    if isinstance(attn_2d, torch.Tensor):
        attn_t = attn_2d.detach().float().unsqueeze(0).unsqueeze(0)  # [1,1,gh,gw] -> gray image
    else:
        attn_t = torch.as_tensor(attn_2d, dtype=torch.float32).unsqueeze(0).unsqueeze(0)  # [1,1,gh,gw] -> gray image
    attn_up = F.interpolate(attn_t, size=(out_h, out_w), mode="bilinear", align_corners=False)[0,0] # [32, 32]
    attn_up = attn_up - attn_up.min()
    attn_up = attn_up / (attn_up.max() + 1e-8)
    return attn_up.cpu().numpy()  # (out_h,out_w)

def _get_attention_maps_for_image(model, x1, vit_cfg, layer_idx=None, head_idx=None):
    """단일 이미지(x1)에 대해 attention map(overlay용)을 준비합니다.
    - layer_idx=None 또는 layer_idx==0 이면 '맨 마지막 레이어'로 자동 설정합니다(해석 용이).
    - head_idx=None 이면 'head 평균'을 반환.
    - head_idx가 정수면 해당 head만 반환.
    반환: (maps, used_layer_idx)
      - maps: list of (title, attn_up_HxW)
    """
    ???
    with torch.no_grad():
        _ = model(x1)

    # (중요) 기본값: 마지막 레이어를 사용
    depth = int(vit_cfg.depth)
    used_layer_idx = (depth - 1) if (layer_idx is None or layer_idx == 0) else int(layer_idx)

    attn_bhNN = _extract_last_attn(model, used_layer_idx)  # [1,H,N,N]

    # patch grid 크기 = (H/patch, W/patch)
    ps = int(vit_cfg.patch_size) if not isinstance(vit_cfg.patch_size, tuple) else int(vit_cfg.patch_size[0])
    # 입력 이미지 크기 (CIFAR-10: 32x32, MNIST: HxW 등)에서 patch grid 크기를 계산합니다.
    if isinstance(vit_cfg.image_size, tuple):
        img_h, img_w = int(vit_cfg.image_size[0]), int(vit_cfg.image_size[1])
    else:
        img_h, img_w = int(vit_cfg.image_size), int(vit_cfg.image_size)

    grid_h, grid_w = img_h // ps, img_w // ps

    maps = []
    if head_idx is None:
        # head 평균 1장
        cls_1d = _cls_to_patch_map(attn_bhNN, head_idx=None)
        attn_2d = _to_2d_grid(cls_1d, grid_h, grid_w)
        maps.append(("MEAN", _upsample_to_image(attn_2d, out_h=img_h, out_w=img_w)))
    else:
        # 특정 head 1장
        cls_1d = _cls_to_patch_map(attn_bhNN, head_idx=int(head_idx))
        attn_2d = _to_2d_grid(cls_1d, grid_h, grid_w)
        maps.append((f"H{int(head_idx)}", _upsample_to_image(attn_2d, out_h=img_h, out_w=img_w)))

    return maps, used_layer_idx

def show_attention_overlays(
    model,
    test_set,
    vit_cfg,
    device,
    n_samples: int = 6,
    layer_idx=None,
    head_idx=None,
    show_all_heads: bool = True,
    sample_mode: str = "correct",  # "correct" | "incorrect" | "all"
    seed: int = 0,
):
    """여러 이미지에 대해 ORG(원본) + attention overlay를 나란히 보여줍니다.

    Args:
        model: 학습된 ViT 모델
        test_set: (image, label) 형태의 dataset
        vit_cfg: ViTConfig (depth, heads, patch_size 등)
        device: torch.device
        n_samples: 출력할 샘플 개수
        layer_idx: 보고 싶은 encoder 블록 인덱스
            - None 또는 0이면 마지막 레이어(depth-1)로 자동 설정(권장)
        head_idx: 보고 싶은 head 인덱스
            - None이면 head 평균(MEAN)
        show_all_heads: True면 마지막 레이어의 모든 head(H0..H{heads-1})도 함께 출력
        sample_mode: "correct"/"incorrect"/"all" 로 샘플 필터링
        seed: 랜덤 샘플링 재현성

    출력:
        각 샘플마다 [ORG] + [MEAN] + [H0..] 형태로 한 줄(row)로 표시합니다.
    """
    random.seed(seed)

    # 1) 샘플 후보 인덱스 준비
    indices = list(range(len(test_set)))

    # 2) 필요 시: 샘플 필터링 (correct/incorrect/all)
    # sample_mode에 따라 후보를 거릅니다.
    # - "correct"   : 정답(=맞춘) 샘플만
    # - "incorrect" : 오답(=틀린) 샘플만
    # - "all"       : 전체(필터 없음)
    sample_mode = (sample_mode or "all").lower().strip()
    if sample_mode not in {"correct", "incorrect", "all"}:
        print(f"[warn] sample_mode={sample_mode} 는 지원하지 않습니다. -> 'all'로 처리합니다.")
        sample_mode = "all"

    if sample_mode in {"correct", "incorrect"}:
        filtered = []
        ???
        with torch.no_grad():
            for idx in indices:
                x, y = test_set[idx]
                x1 = x.unsqueeze(0).to(device)  # (1, 3, 32, 32)
                logits = ???
                pred = ???
                is_correct = (pred == int(y))
                if (sample_mode == "correct" and is_correct) or (sample_mode == "incorrect" and (not is_correct)):
                    filtered.append(idx)

        indices = filtered
        if len(indices) == 0:
            msg = "정답" if sample_mode == "correct" else "오답"
            print(f"[warn] 조건에 맞는 {msg} 샘플이 없습니다. (모델 학습 상태/seed/n_samples를 확인하세요.)")
            return

    # 3) 샘플링
    chosen = random.sample(indices, k=min(n_samples, len(indices))) if len(indices) > 0 else []

    # 4) 시각화: 샘플마다 ORG + (MEAN + Heads) 를 한 row로 출력
    heads = int(vit_cfg.heads)
    for idx in chosen:
        x, y = test_set[idx]
        label = int(y)

        x1 = x.unsqueeze(0).to(device)
        ???
        with torch.no_grad():
            logits = ???
            pred = ???

        # (A) 평균 attention(MEAN)
        mean_maps, used_layer = _get_attention_maps_for_image(
            model=model, x1=x1, vit_cfg=vit_cfg,
            layer_idx=layer_idx, head_idx=None
        )

        # (B) 각 head attention(H0..)
        head_maps = []
        if show_all_heads:
            for h in range(heads):
                hm, _ = _get_attention_maps_for_image(
                    model=model, x1=x1, vit_cfg=vit_cfg,
                    layer_idx=layer_idx, head_idx=h
                )
                head_maps.extend(hm)  # hm은 1개짜리 리스트

        # 표시할 컬럼 구성: ORG + MEAN + heads...
        maps_to_show = mean_maps + head_maps

        ncols = 1 + len(maps_to_show)
        fig, axes = plt.subplots(1, ncols, figsize=(3.2 * ncols, 3.2))

        # ORG(원본)
        ax0 = axes[0]
        imshow_chw(ax0, x, title=None)
        ax0.axis("off")
        ax0.set_title(f"ORG | GT={class_names[int(label)]} / Pred={class_names[int(pred)]}\n(layer={used_layer})")

        # Overlays
        for j, (title, attn_up) in enumerate(maps_to_show, start=1):
            ax = axes[j]
            base = chw_to_hwc_np(x)  # (H,W,3), 0~1
            ax.imshow(base)
            ax.imshow(attn_up, alpha=0.45)  # heatmap overlay (컬러맵 기본값 사용)
            ax.axis("off")
            ax.set_title(f"{title}")

        plt.tight_layout()
        plt.show()

# 사용 예시:
# - 기본값(layer_idx=None): 마지막 레이어의 MEAN + 모든 head를 출력
# - sample_mode="incorrect": 오답 샘플만 골라서 보면 해석이 더 쉬운 경우가 많습니다.
show_attention_overlays(
    model=model,
    test_set=test_set,
    vit_cfg=vit_cfg,
    device=device,
    n_samples=6,
    layer_idx=None,           # None 또는 0이면 마지막 레이어를 사용
    head_idx=None,            # None이면 평균(MEAN)
    show_all_heads=True,      # 모든 head도 함께 보기
    sample_mode="correct",
    seed=0,
)

#### 정답

??? 순서: `model.eval()`, `model.eval()`, `model(x1)`, `int(logits.argmax(dim=1).item())`, `model.eval()`, `model(x1)`, `int(logits.argmax(dim=1).item())`

#### 간략 해설

1. `model.eval()`: 평가/추론에서는 Dropout을 끄고 BatchNorm running statistics를 사용해야 합니다.
2. `model.eval()`: 평가/추론에서는 Dropout을 끄고 BatchNorm running statistics를 사용해야 합니다.
3. `model(x1)`: forward pass에 해당하는 핵심 코드로, 이 위치의 입력/출력 흐름을 결정합니다.
4. `int(logits.argmax(dim=1).item())`: 가장 큰 logit 위치가 모델이 선택한 class이므로 top-1 예측에 사용합니다.
5. `model.eval()`: 평가/추론에서는 Dropout을 끄고 BatchNorm running statistics를 사용해야 합니다.
6. `model(x1)`: forward pass에 해당하는 핵심 코드로, 이 위치의 입력/출력 흐름을 결정합니다.
7. `int(logits.argmax(dim=1).item())`: 가장 큰 logit 위치가 모델이 선택한 class이므로 top-1 예측에 사용합니다.


### 문제 14. 7) Confusion Matrix로 오분류 패턴 보기
- 원본 파일/셀: `02_ViT_CIFAR10.ipynb` / code cell index `26`
- 목적: 목적:** 단순 정확도(accuracy)만으로는 보이지 않는 **클래스별 오분류 경향**을 확인합니다.
- 관찰 포인트:
  - **관찰 포인트:**
  - 어떤 클래스 쌍(예: `cat↔dog`, `automobile↔truck`, `deer↔horse`)이 자주 헷갈리는지
  - 특정 클래스에서만 오분류가 집중되는지(데이터/모델 편향 가능성)
  - 이후 `오분류 샘플 시각화`와 함께 보면 “왜” 틀렸는지 해석이 쉬워집니다.
- 출제 포인트:
  - 모델 결과와 정답을 이용한 학습
- 빈칸 기준:
  - eval mode
  - forward pass


In [ ]:
def plot_confusion_matrix(model, data_loader):
    ???
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for data, target in data_loader:
            data = data.to(device)
            target = target.to(device)
            output = ???  # 입력 배치에 대한 모델 forward(로짓 출력)
            _, preds = torch.max(output, dim=1)   # preds = maximum indices
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(target.cpu().numpy())

    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)

    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.show()

plot_confusion_matrix(model, test_loader)


#### 정답

??? 순서: `model.eval()`, `model(data)`

#### 간략 해설

1. `model.eval()`: 평가/추론에서는 Dropout을 끄고 BatchNorm running statistics를 사용해야 합니다.
2. `model(data)`: forward pass에 해당하는 핵심 코드로, 이 위치의 입력/출력 흐름을 결정합니다.


### 문제 15. 8) 오분류(실패) 샘플만 시각화하기
- 원본 파일/셀: `02_ViT_CIFAR10.ipynb` / code cell index `28`
- 목적: 목적:** 단순히 "맞춘 샘플"을 보는 것보다, **틀린 샘플(오분류)** 을 모아 보면 모델이 약한 패턴을 빠르게 파악할 수 있습니다.
- 출제 포인트:
  - 모델 결과와 정답을 이용한 학습
  - 평가 metric 및 예측 후처리
- 빈칸 기준:
  - eval mode
  - forward pass
  - 예측 class 선택


In [ ]:
# 오분류(실패) 샘플만 시각화
def plot_misclassified_samples(model, data_loader, max_samples: int = 10):
    """테스트 데이터에서 오분류된 샘플만 골라 시각화합니다.

    Args:
        model: 학습된 분류 모델
        data_loader: (data, target)을 내는 DataLoader
        max_samples: 최대 출력 샘플 수 (기본 10)
    """
    ???
    mis_images, mis_labels, mis_preds = [], [], []

    with torch.no_grad():
        for data, target in data_loader:
            data = data.to(device)
            target = target.to(device)

            logits = ???                 # [B, num_classes] 로짓
            preds = ???         # 각 샘플의 예측 클래스

            wrong_mask = preds.ne(target)        # 오분류 위치만 True
            if wrong_mask.any():
                # 오분류된 샘플만 추출
                wrong_data = data[wrong_mask].detach().cpu()  # keep as Tensor for denormalize
                wrong_target = target[wrong_mask].detach().cpu()
                wrong_preds = preds[wrong_mask].detach().cpu()

                for x, y, p in zip(wrong_data, wrong_target, wrong_preds):
                    mis_images.append(x)
                    mis_labels.append(int(y))
                    mis_preds.append(int(p))
                    if len(mis_images) >= max_samples:
                        break

            if len(mis_images) >= max_samples:
                break

    if len(mis_images) == 0:
        print("오분류 샘플이 없습니다. (현재 설정/에폭에서 테스트를 모두 맞췄을 가능성이 있습니다)")
        return

    # 시각화 그리드: 최대 10개를 2×5로 보기 좋게 배치
    n = min(len(mis_images), max_samples)
    rows = 2 if n > 5 else 1
    cols = 5 if n > 5 else n

    fig, axes = plt.subplots(rows, cols, figsize=(3*cols, 3.2*rows))
    axes = np.array(axes).reshape(-1)  # axes를 1D로 평탄화

    for i in range(rows * cols):
        ax = axes[i]
        if i < n:
            img_chw = mis_images[i]  # CIFAR-10: (3,32,32)
            img_vis = denormalize_cifar10(img_chw).detach().cpu().numpy().transpose(1, 2, 0)  # HWC
            ax.imshow(img_vis)
            ax.set_title(f"GT: {class_names[mis_labels[i]]} / Pred: {class_names[mis_preds[i]]}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()


# 실행: 오분류 샘플만 출력
plot_misclassified_samples(model, test_loader, max_samples=10)


#### 정답

??? 순서: `model.eval()`, `model(data)`, `logits.argmax(dim=1)`

#### 간략 해설

1. `model.eval()`: 평가/추론에서는 Dropout을 끄고 BatchNorm running statistics를 사용해야 합니다.
2. `model(data)`: forward pass에 해당하는 핵심 코드로, 이 위치의 입력/출력 흐름을 결정합니다.
3. `logits.argmax(dim=1)`: 가장 큰 logit 위치가 모델이 선택한 class이므로 top-1 예측에 사용합니다.


## 03_DETR.ipynb

### 문제 16. 2) 입력 전처리 및 박스 좌표 후처리
- 원본 파일/셀: `03_DETR.ipynb` / code cell index `7`
- 목적: 목적:** DETR 입력 규격에 맞게 이미지를 전처리하고, 모델이 예측한 박스를 원본 이미지 좌표계로 되돌리는 함수를 준비합니다.
- 관찰 포인트:
  - **관찰 포인트**
  - `T.Compose([...])`는 **Resize/Normalize** 등 입력 변환을 순차적으로 적용함
  - DETR의 박스는 보통 정규화 좌표(`cx, cy, w, h`) 형태이므로, 시각화를 위해 **픽셀 좌표**로 스케일 복원이 필요함
  - 후처리 함수는 (정규화 → 픽셀) 변환과 `(cx,cy,w,h) → (x1,y1,x2,y2)` 변환 역할을 담당함
- 출제 포인트:
  - 모델 layer/구조의 의미와 동작 원리
  - 학습/평가 의도에 맞는 데이터 구성과 이미지 변형
- 빈칸 기준:
  - transform 순차 구성
  - 입력 크기 resize
  - tensor 변환
  - 채널 정규화
  - box format conversion
  - tensor stack
  - box format conversion


In [ ]:
# [DETR] 입력 전처리 + 박스 좌표 후처리 함수
# - DETR은 COCO로 학습된 ResNet backbone을 사용하므로, ImageNet mean/std 정규화를 맞춰야 합니다.
# - 모델 출력 박스는 (cx, cy, w, h) 형식이며 0~1로 정규화되어 있으므로, (xmin, ymin, xmax, ymax)로 변환 후 원본 이미지 크기로 rescale 합니다.

# standard PyTorch mean-std input image normalization
transform = ???([  # 전처리 파이프라인 정의(논문/코드: data augmentation과 정규화)
    ???,  # 짧은 변을 800에 맞추는 리사이즈(공식 데모 설정)
    ???,  # PIL → torch.Tensor (C,H,W), 0~1 스케일
    ???  # ImageNet mean/std로 정규화(backbone 사전학습 분포와 정합)
])

# for output bounding box post-processing
def ???:  # (cx,cy,w,h) → (xmin,ymin,xmax,ymax) 변환
    x_c, y_c, w, h = x.unbind(1)  # 각 성분을 분리: x.shape=(N,4) 가정
    b = [(x_c - 0.5 * w), (y_c - 0.5 * h),  # 중심좌표에서 좌상/우하 꼭짓점 계산
         (x_c + 0.5 * w), (y_c + 0.5 * h)]
    return ???  # (N,4)로 다시 스택

def rescale_bboxes(out_bbox, size):  # 정규화 박스를 픽셀 좌표로 변환
    img_w, img_h = size  # PIL size = (W,H)
    b = ???  # 우선 xyxy로 변환
    b = b * torch.tensor([img_w, img_h, img_w, img_h], dtype=torch.float32)  # (W,H,W,H) 스케일로 곱해 픽셀 단위로 확장
    return b


#### 정답

??? 순서: `T.Compose`, `T.Resize(800)`, `T.ToTensor()`, `T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])`, `box_cxcywh_to_xyxy(x)`, `torch.stack(b, dim=1)`, `box_cxcywh_to_xyxy(out_bbox)`

#### 간략 해설

1. `T.Compose`: 여러 transform을 지정한 순서대로 적용해야 하므로 사용합니다.
2. `T.Resize(800)`: 입력 크기 resize에 해당하는 핵심 코드로, 이 위치의 입력/출력 흐름을 결정합니다.
3. `T.ToTensor()`: PIL 이미지를 PyTorch가 처리하는 C,H,W tensor로 바꿉니다.
4. `T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])`: 채널별 평균과 표준편차로 입력 분포를 맞춰 학습을 안정화합니다.
5. `box_cxcywh_to_xyxy(x)`: DETR의 중심좌표 박스를 시각화 가능한 좌상/우하 좌표 형식으로 변환합니다.
6. `torch.stack(b, dim=1)`: DETR의 중심좌표 박스를 시각화 가능한 좌상/우하 좌표 형식으로 변환합니다.
7. `box_cxcywh_to_xyxy(out_bbox)`: DETR의 중심좌표 박스를 시각화 가능한 좌상/우하 좌표 형식으로 변환합니다.


### 문제 17. 4) 사전학습 DETR 모델 로드
- 원본 파일/셀: `03_DETR.ipynb` / code cell index `11`
- 목적: 목적:** `torch.hub`를 이용해 COCO로 사전학습된 DETR-ResNet50 모델을 로드합니다.
- 관찰 포인트:
  - **관찰 포인트**
  - `pretrained=True`로 로드된 모델은 **바로 추론**이 가능함
  - DETR은 **NMS 없이** set prediction 형태로 결과를 내도록 학습된 모델임
  - 이후 셀에서 출력되는 `pred_logits`, `pred_boxes`의 shape가 `num_queries`에 의해 결정됨
- 출제 포인트:
  - 핵심 API와 입력/출력 shape 이해
- 빈칸 기준:
  - pretrained model load


In [ ]:
# [DETR] 사전학습 모델 로드
# - 공식 구현(facebookresearch/detr)의 pretrained 모델을 torch.hub로 로드합니다.
# - backbone=ResNet-50, transformer encoder-decoder + object queries 구조.

model = ???('facebookresearch/detr', 'detr_resnet50', pretrained=True)  # 가중치 포함 pretrained DETR-ResNet50 로드
model.eval();  # Dropout/BN 등을 추론 모드로 전환

#### 정답

??? 순서: `torch.hub.load`

#### 간략 해설

1. `torch.hub.load`: 사전학습된 DETR 모델을 불러와 객체 탐지 추론에 사용합니다.


### 문제 18. 5) 모델 구조 출력
- 원본 파일/셀: `03_DETR.ipynb` / code cell index `13`
- 목적: 목적:** DETR 모델의 큰 구성(backbone, transformer, prediction head)을 출력하여 구조를 한 번 확인합니다.
- 관찰 포인트:
  - **관찰 포인트**
  - backbone(CNN)이 spatial feature map을 만들고, transformer가 query 기반으로 객체 표현을 형성함
  - 마지막 head가 `class logits`와 `box`를 각각 예측함
  - 출력 구조를 보면 **어텐션을 어디서 뽑을지(hook 지점)**를 설계할 수 있음
- 출제 포인트:
  - 핵심 API와 입력/출력 shape 이해
- 빈칸 기준:
  - pretrained model load


In [ ]:
# ============================================================
# DETR 전체 모델 구조 출력 (모델 로드 직후)
# ------------------------------------------------------------
# ??? 로 불러온 detr_resnet50 모델의
# PyTorch Module 계층 구조를 출력합니다.
#
# 논문 관점 대응:
# - backbone  → CNN feature extractor (ResNet-50)
# - transformer → Encoder / Decoder (object queries)
# - class_embed / bbox_embed → prediction heads
# ============================================================

print("=== DETR Model Architecture ===")
print(model)


#### 정답

??? 순서: `torch.hub.load`

#### 간략 해설

1. `torch.hub.load`: 사전학습된 DETR 모델을 불러와 객체 탐지 추론에 사용합니다.


### 문제 19. 7) 한 장 이미지 추론 및 필터링
- 원본 파일/셀: `03_DETR.ipynb` / code cell index `17`
- 목적: 목적:** 이미지 1장을 DETR에 입력하여 `pred_logits`, `pred_boxes`를 얻고, 신뢰도 기준으로 예측을 선택합니다.
- 관찰 포인트:
  - **관찰 포인트**
  - `outputs`는 dict 형태이며 대표 키는 `pred_logits`, `pred_boxes`
  - `pred_logits` shape: `(B, num_queries, num_classes+1)` (**+1은 no-object**)
  - `pred_boxes` shape: `(B, num_queries, 4)` (정규화된 박스)
- 출제 포인트:
  - 모델 결과와 정답을 이용한 학습
  - 평가 metric 및 예측 후처리
- 빈칸 기준:
  - forward pass
  - confidence filtering


In [ ]:
# [DETR] 한 장 이미지 추론 + 신뢰도 필터링 + 박스 스케일 복원
# - outputs['pred_logits']: (1, num_queries, num_classes+1) 로짓. 마지막 '+1'은 'no-object' 클래스.
# - outputs['pred_boxes']: (1, num_queries, 4) 박스 (cx,cy,w,h) 정규화 좌표.
# - 논문처럼 모든 쿼리가 박스를 내지만, 여기서는 시각화를 위해 높은 confidence만 남깁니다.

# mean-std normalize the input image (batch-size: 1)
img = transform(im).unsqueeze(0)  # 전처리 후 배치 차원 추가: (1,3,H,W)

from torchinfo import summary
model_summary = summary(model,
        input_data=img,
        col_names=["input_size", "output_size", "num_params"],
        depth=3)
print(model_summary)

# propagate through the model
outputs = ???  # DETR forward → 로짓/박스 예측

# print keys of outputs
for key in outputs.keys():
    # value를 가져와서 shape 출력
    shape = outputs[key].shape
    print(f"Key : {key}")
    print(f"Shape   : {shape}")
    print("-" * 30)  # 구분선 추가

# keep only predictions with 0.9+ confidence
probas = outputs['pred_logits'].softmax(-1)[0, :, :-1]  # 쿼리별 클래스 확률 계산(no-object 제외): shape=(num_queries,num_classes)
keep = ???  # 각 쿼리의 max class score가 임계값을 넘는 것만 선택
print(f"keep Shape : {keep.shape}")


# convert boxes from [0; 1] to image scales
bboxes_scaled = rescale_bboxes(outputs['pred_boxes'][0, keep], im.size)  # 선택된 박스를 픽셀 좌표로 변환
print(f"bboxes_scaled Shape : {bboxes_scaled.shape}")


#### 정답

??? 순서: `model(img)`, `probas.max(-1).values > 0.9`

#### 간략 해설

1. `model(img)`: forward pass에 해당하는 핵심 코드로, 이 위치의 입력/출력 흐름을 결정합니다.
2. `probas.max(-1).values > 0.9`: confidence filtering에 해당하는 핵심 코드로, 이 위치의 입력/출력 흐름을 결정합니다.


### 문제 20. 9) 내부 텐서 추출을 위한 forward hook
- 원본 파일/셀: `03_DETR.ipynb` / code cell index `21`
- 목적: 목적:** DETR의 중간 특징맵(conv features)과 encoder/decoder attention weight를 캡처하기 위해 hook을 등록합니다.
- 관찰 포인트:
  - **관찰 포인트**
  - `register_forward_hook`은 **forward 중간 출력**을 저장하는 전형적인 디버깅/분석 기법
  - 어떤 레이어에 hook을 걸었는지에 따라 얻는 텐서의 의미가 크게 달라짐
  - 이후 attention 시각화는 이 셀에서 저장한 텐서(conv/enc_attn/dec_attn)에 의존함
- 출제 포인트:
  - 모델 결과와 정답을 이용한 학습
- 빈칸 기준:
  - forward pass


In [ ]:
# [DETR] 내부 텐서(특징맵/어텐션) 추출을 위한 forward hook
# - 논문 분석 포인트:
#   1) backbone CNN 특징맵(conv_features)
#   2) 마지막 encoder layer의 self-attention(enc_attn_weights)
#   3) 마지막 decoder layer의 cross-attention(dec_attn_weights)
# - PyTorch hook을 등록해 forward 중간 결과를 캡처합니다.

# use lists to store the outputs via up-values
conv_features, enc_attn_weights, dec_attn_weights = [], [], []  # hook 결과를 담을 리스트(클로저 변수) 초기화

hooks = [  # 각 모듈에 forward hook 등록
    model.backbone[-2].register_forward_hook(  # CNN backbone의 특정 stage 출력(feature map) 저장
        lambda self, input, output: conv_features.append(output) # lambda 인자들 : 실행문
    ),
    model.transformer.encoder.layers[-1].self_attn.register_forward_hook(  # encoder self-attention의 attention weights(output[1]) 저장
        lambda self, input, output: enc_attn_weights.append(output[1])   # self-attention weight -> output[1]
    ),
    model.transformer.decoder.layers[-1].multihead_attn.register_forward_hook(  # decoder cross-attention의 attention weights(output[1]) 저장
        lambda self, input, output: dec_attn_weights.append(output[1])   # cross-attention weight -> output[1]
    ),
]

# propagate through the model
outputs = ???  # hook이 등록된 상태로 forward 실행

for hook in hooks:  # 등록했던 hook 해제(메모리/부작용 방지)
    hook.remove()

# don't need the list anymore
conv_features = conv_features[0]  # hook 결과를 담을 리스트(클로저 변수) 초기화
print(f"features shape: {conv_features['0'].tensors.shape}")   # [1, channels, H, W]
enc_attn_weights = enc_attn_weights[0]  # encoder attention weights 텐서
print(f"enc_attn_weights shape: {enc_attn_weights.shape}")   # [1, HW, HW] -> 각 feature간 attention
dec_attn_weights = dec_attn_weights[0]  # decoder attention weights 텐서
print(f"dec_attn_weights shape: {dec_attn_weights.shape}")   # [1, N, HW] -> 각 객체와 feature간 attention


#### 정답

??? 순서: `model(img)`

#### 간략 해설

1. `model(img)`: forward pass에 해당하는 핵심 코드로, 이 위치의 입력/출력 흐름을 결정합니다.


### 문제 21. 14) AttentionVisualizer (Encoder Self-attention) 클래스
- 원본 파일/셀: `03_DETR.ipynb` / code cell index `31`
- 목적: 목적:** 이미지 내 특정 지점(Reference Point)을 기준으로 **Encoder가 전역적인 문맥(Global Context)을 어떻게 수집하는지** 확인하는 인터랙티브 도구를 정의합니다.
- 관찰 포인트:
  - **관찰 포인트**
  - **Encoder Self-attention의 역할:** 이미지의 한 픽셀(Query)이 '자기 자신을 정의하기 위해' 이미지 전체의 다른 픽셀들(Keys)을 얼마나 참고하는지 나타냄
  - **인터랙티브 시각화:** 슬라이더를 통해 이미지 위의 **빨간 점(Query 좌표)**을 옮기면, 해당 위치의 토큰이 바라보는 **Attention Map**이 실시간으로 업데이트됨.
  - **객체 실루엣 인지:** 빨간 점을 객체(예: 고양이) 위에 두면 어텐션이 해당 객체의 전체적인 형태를 따라 활성화되는 것을 볼 수 있음. 이는 인코더가 이미 객체와 배경을 분리하여 인지하고 있음을 보여줌.
- 출제 포인트:
  - 모델 결과와 정답을 이용한 학습
- 빈칸 기준:
  - forward pass


In [ ]:
# [DETR] AttentionVisualizer 클래스
# - 목적: 이미지 내 특정 지점(Pixel)을 기준으로 Encoder의 'Self-attention'이 어떻게 전역 문맥을 섞는지 직관적으로 이해.
# - 주요 아이디어(논문 구현 관점):
#   1) Backbone -> Transformer Encoder: 이미지를 토큰화하고 각 토큰(픽셀)들 간의 관계를 Self-attention으로 연산.
#   2) 사용자가 슬라이더로 이미지 상의 특정 좌표(Query 지점)를 선택.
#   3) 해당 좌표의 토큰이 이미지 전체(Key 지점들)를 얼마나 참고하는지 마지막 Encoder 레이어에서 추출하여 시각화.
# - 아래 코드는 forward hook으로 Encoder 내부 텐서를 캡처하고, 인터랙티브 위젯을 통해 특정 위치의 어텐션 맵을 그립니다.

class AttentionVisualizer:  # DETR 내부 어텐션을 시각화하는 유틸 클래스 정의
    def __init__(self, model, transform):  # 모델/전처리/위젯 초기화
        self.model = model  # 외부에서 로드한 DETR 모델 보관
        self.transform = transform  # 입력 이미지 전처리 함수 보관

        self.url = ""
        self.cur_url = None
        self.pil_img = None
        self.tensor_img = None

        self.conv_features = None
        self.enc_attn_weights = None
        self.dec_attn_weights = None

        self.setup_widgets()

    def setup_widgets(self):
        self.sliders = [
            widgets.Text(
                value='http://images.cocodataset.org/val2017/000000039769.jpg',
                placeholder='Type something',
                description='URL (ENTER):',
                disabled=False,
                continuous_update=False,
                layout=widgets.Layout(width='100%')
            ),
            widgets.FloatSlider(min=0, max=0.99,
                        step=0.02, description='X coordinate', value=0.72,
                        continuous_update=False,
                        layout=widgets.Layout(width='50%')
                        ),
            widgets.FloatSlider(min=0, max=0.99,
                        step=0.02, description='Y coordinate', value=0.40,
                        continuous_update=False,
                        layout=widgets.Layout(width='50%')),
            widgets.Checkbox(
              value=False,
              description='Direction of self attention',
              disabled=False,
              indent=False,
              layout=widgets.Layout(width='50%'),
          ),
            widgets.Checkbox(
              value=True,
              description='Show red dot in attention',
              disabled=False,
              indent=False,
              layout=widgets.Layout(width='50%'),
          )
        ]
        self.o = widgets.Output()  # matplotlib 출력이 들어갈 영역

    def compute_features(self, img):  # hook을 등록해 feature/attention을 수집한 뒤 forward 수행
        model = self.model  # 외부에서 로드한 DETR 모델 보관
        # use lists to store the outputs via up-values
        conv_features, enc_attn_weights, dec_attn_weights = [], [], []

        hooks = [
            model.backbone[-2].register_forward_hook(  # backbone feature map 캡처
                lambda self, input, output: conv_features.append(output)
            ),
            model.transformer.encoder.layers[-1].self_attn.register_forward_hook(  # encoder 마지막 layer self-attention 캡처
                lambda self, input, output: enc_attn_weights.append(output[1])
            ),
            model.transformer.decoder.layers[-1].multihead_attn.register_forward_hook(  # decoder 마지막 layer cross-attention 캡처
                lambda self, input, output: dec_attn_weights.append(output[1])
            ),
        ]
        # propagate through the model
        outputs = ???  # hook이 걸린 상태로 DETR 추론 실행

        for hook in hooks:  # hook 해제(중복 캡처 방지)
            hook.remove()

        # don't need the list anymore
        self.conv_features = conv_features[0]  # 리스트에서 실제 텐서/NestedTensor 꺼내기
        self.dec_attn_weights = dec_attn_weights[0]
        # get the HxW shape of the feature maps of the CNN
        shape = self.conv_features['0'].tensors.shape[-2:]
        # and reshape the self-attention to a more interpretable shape
        self.enc_attn_weights = enc_attn_weights[0].reshape(shape + shape)

    def compute_on_image(self, url):
        if url != self.url:
            self.url = url
            self.pil_img = Image.open(requests.get(url, stream=True).raw)
            # mean-std normalize the input image (batch-size: 1)
            self.tensor_img = self.transform(self.pil_img).unsqueeze(0)  # 입력 이미지 전처리 함수 보관
            self.compute_features(self.tensor_img)

    def update_chart(self, change):
        with self.o:  # Output 영역에 그림을 그리기 위해 컨텍스트 진입
            clear_output()  # 이전 출력 삭제 후 새 결과 렌더링

            # j and i are the x and y coordinates of where to look at
            # sattn_dir is which direction to consider in the self-attention matrix
            # sattn_dot displays a red dot or not in the self-attention map
            url, j, i, sattn_dir, sattn_dot = [s.value for s in self.sliders]

            fig, axs = plt.subplots(ncols=2, nrows=1, figsize=(9, 4))
            self.compute_on_image(url)

            # convert reference point to absolute coordinates
            j = int(j * self.tensor_img.shape[-1])
            i = int(i * self.tensor_img.shape[-2])

            # how much was the original image upsampled before feeding it to the model
            scale = self.pil_img.height / self.tensor_img.shape[-2]

            # compute the downsampling factor for the model
            # it should be 32 for standard DETR and 16 for DC5
            sattn = self.enc_attn_weights
            fact = 2 ** round(math.log2(self.tensor_img.shape[-1] / sattn.shape[-1]))

            # round the position at the downsampling factor
            x = ((j // fact) + 0.5) * fact
            y = ((i // fact) + 0.5) * fact

            axs[0].imshow(self.pil_img)
            axs[0].axis('off')
            axs[0].add_patch(plt.Circle((x * scale, y * scale), fact // 2, color='r'))

            idx = (i // fact, j // fact)

            if sattn_dir:
                sattn_map = sattn[idx[0], idx[1], ...]
            else:
                sattn_map = sattn[..., idx[0], idx[1]]

            axs[1].imshow(sattn_map, cmap='cividis', interpolation='nearest')  # attention heatmap 출력
            if sattn_dot:
                axs[1].add_patch(plt.Circle((idx[1],idx[0]), 1, color='r'))
            axs[1].axis('off')
            axs[1].set_title(f'self-attention{(i, j)}')

            plt.show()

    def run(self):
      for s in self.sliders:
          s.observe(self.update_chart, 'value')
      self.update_chart(None)
      url, x, y, d, sattn_d = self.sliders
      res = widgets.VBox(
      [
          url,
          widgets.HBox([x, y]),
          widgets.HBox([d, sattn_d]),
          self.o
      ])
      return res


#### 정답

??? 순서: `model(img)`

#### 간략 해설

1. `model(img)`: forward pass에 해당하는 핵심 코드로, 이 위치의 입력/출력 흐름을 결정합니다.


## 04_Unet.ipynb

### 문제 22. 2) U-Net 모델 구현
- 원본 파일/셀: `04_Unet.ipynb` / code cell index `6`
- 목적: 목적:** U-Net의 표준 블록을 구현하고, Encoder/Decoder의 연결 구조를 코드로 확인합니다.
- 관찰 포인트:
  - **관찰 포인트**
  - `DoubleConv`(Conv–BN–ReLU 반복)이 feature 추출에 어떻게 쓰이는지
  - Down path에서 **Pooling/Stride**로 해상도가 줄어드는 지점
  - Up path에서 **Up-sampling(ConvTranspose2d 또는 bilinear)** 이 적용되는 지점
- 출제 포인트:
  - 모델 layer/구조의 의미와 동작 원리
- 빈칸 기준:
  - Conv2d layer
  - ReLU activation
  - Conv2d layer
  - ReLU activation
  - max pooling
  - upsampling layer
  - skip connection concat
  - Conv2d layer


In [ ]:
# =========================
# 3) U-Net 모델 정의
# =========================
# 이 섹션의 목표:
# - U-Net의 "인코더(Down) → 보틀넥 → 디코더(Up) + Skip Connection" 구조를
#   PyTorch 코드로 직접 따라가며 이해합니다.
#
# 핵심 아이디어(한 줄):
# - Down에서 공간 해상도(H,W)는 줄이고 채널(C)은 늘리면서 특징을 추출하고,
#   Up에서 해상도를 복원하면서 Down 단계의 특징맵을 Skip으로 concat하여
#   localization(위치 정보)을 되살립니다.

class DoubleConv(nn.Module):
    """U-Net 기본 블록: (Conv → ReLU) × 2

    - 첫 번째 Conv가 채널을 '중간 채널(mid_channels)'로 바꾸고,
      두 번째 Conv가 '출력 채널(out_channels)'로 맞춥니다.
    - 기본값(mid_channels=None)일 때는 mid_channels=out_channels로 두어,
      (C_in → C_out → C_out) 형태가 됩니다.

    입력/출력 텐서 형태:
      - 입력:  (B, C_in, H, W)
      - 출력:  (B, C_out, H, W)  # padding=1 이라 H,W 유지
    """
    def __init__(self, in_channels: int, out_channels: int, mid_channels: int | None = None):
        super().__init__()

        # U-Net에서 업샘플 후 concat을 하면 채널이 2배가 되므로,
        # bilinear 업샘플링을 쓸 때는 mid_channels=in_channels//2 처럼
        # '중간 채널'을 줄여주는 방식이 흔히 사용됩니다.
        if mid_channels is None:
            mid_channels = out_channels

        self.net = nn.Sequential(
            # 1) (C_in → C_mid)
            ???,
            ???,

            # 2) (C_mid → C_out)
            ???,
            ???,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

class Down(nn.Module):
    """Downsampling 블록: MaxPool(2)로 해상도 1/2 → DoubleConv

    입력/출력 텐서 형태:
      - 입력:  (B, C_in,  H,  W)
      - 출력:  (B, C_out, H/2, W/2)
    """
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.pool = ???
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 1) 해상도 축소
        x = self.pool(x)           # (B, C_in, H/2, W/2)
        # 2) 채널 확장 + 특징 추출
        x = self.conv(x)           # (B, C_out, H/2, W/2)
        return x


class Up(nn.Module):
    """Upsampling 블록: 업샘플링 → (Skip concat) → DoubleConv

    구현 관점(중요):
    - in_channels 는 concat 이후 채널 수를 의미합니다.
      예) x1(디코더) 채널=512, x2(skip) 채널=512 → concat 채널=1024 → in_channels=1024

    - bilinear=True:
        1) 업샘플은 파라미터 없는 bilinear interpolation으로 수행
        2) concat 후 DoubleConv에서 mid_channels를 in_channels//2 로 두어
           채널을 자연스럽게 '절반'으로 줄이는 방식(원 논문/레퍼런스 구현과 동일 계열)

    - bilinear=False:
        ConvTranspose2d로 업샘플 자체를 학습(파라미터 증가)
    """
    def __init__(self, in_channels: int, out_channels: int, bilinear: bool = False):
        super().__init__()

        if bilinear:
            # (B, C, H/2, W/2) → (B, C, H, W)
            self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)

            # bilinear일 때는 업샘플 후 채널 수(C)가 그대로 유지됩니다.
            # concat 결과(in_channels)를 DoubleConv로 처리하되,
            # 첫 conv의 출력(mid_channels)을 in_channels//2로 두어 채널을 줄입니다.
            self.conv = DoubleConv(in_channels, out_channels, mid_channels=in_channels // 2)
        else:
            # (B, C, H/2, W/2) → (B, C/2, H, W)  (deconv가 채널도 절반으로 줄여줌)
            self.up = ???
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        # x1: (B, C_dec, H/2, W/2)  / x2: (B, C_skip, H, W)

        # 1) 업샘플링: 해상도를 skip과 맞추기
        x1 = self.up(x1)  # bilinear: 채널 유지 / deconv: 채널이 절반으로 감소

        # 2) (필요 시) 패딩으로 크기 정렬
        #    - 홀수 크기 입력 등으로 인해 x1과 x2의 H/W가 1~2 픽셀 정도 다를 수 있습니다.
        diff_y = x2.size(2) - x1.size(2)
        diff_x = x2.size(3) - x1.size(3)
        x1 = F.pad(x1, [diff_x // 2, diff_x - diff_x // 2,  # left, right
                        diff_y // 2, diff_y - diff_y // 2]) # top, bottom

        # 3) 채널 방향 concat (skip 연결)
        #    cat dim=1 은 채널(C) 축
        x = ???  # (B, C_skip + C_up, H, W) == (B, in_channels, H, W)

        # 4) conv 블록으로 특징 정제 + 채널 축소
        return self.conv(x)

class OutConv(nn.Module):
    """마지막 1x1 conv: 채널을 클래스 수로 매핑

    예)
      - binary segmentation: n_classes=1 (logits 1채널)
      - multi-class:         n_classes=K (logits K채널)
    """
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.conv = ???

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.conv(x)


class UNet(nn.Module):
    """U-Net 전체 모델

    전형적인 채널 구성 예:
      1 → 64 → 128 → 256 → 512 → 1024 (down)
      1024 → 512 → 256 → 128 → 64 (up)
    """
    def __init__(self, n_channels: int, n_classes: int, bilinear: bool = False):
        super().__init__()

        # -------------------------
        # Encoder (Contracting path)
        # -------------------------
        self.inc = DoubleConv(n_channels, 64)   # (B, n_channels, H, W) → (B, 64, H, W)
        self.down1 = Down(64, 128)              # → (B, 128, H/2, W/2)
        self.down2 = Down(128, 256)             # → (B, 256, H/4, W/4)
        self.down3 = Down(256, 512)             # → (B, 512, H/8, W/8)

        # bilinear 업샘플링이면 파라미터/연산을 줄이기 위해 bottleneck 채널을 1024 대신 512로 줄이는 경우가 많음
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)  # → (B, 1024/f, H/16, W/16)

        # -------------------------
        # Decoder (Expanding path)
        # -------------------------
        # Up 블록의 in_channels는 "concat 후 채널" 기준으로 설계되어야 합니다.
        self.up1 = Up(1024, 512 // factor, bilinear)  # (skip=512)와 concat을 고려
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)

        # -------------------------
        # Output head
        # -------------------------
        self.outc = OutConv(64, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # -------------------------
        # Encoder: skip을 위해 각 단계 출력 저장
        # -------------------------
        x1 = self.inc(x)       # (B, 64, H, W)
        x2 = self.down1(x1)    # (B, 128, H/2, W/2)
        x3 = self.down2(x2)    # (B, 256, H/4, W/4)
        x4 = self.down3(x3)    # (B, 512, H/8, W/8)
        x5 = self.down4(x4)    # (B, 1024/f, H/16, W/16)

        # -------------------------
        # Decoder: 업샘플 + skip concat
        # -------------------------
        x = self.up1(x5, x4)   # (B, 512/f, H/8, W/8)
        x = self.up2(x, x3)    # (B, 256/f, H/4, W/4)
        x = self.up3(x, x2)    # (B, 128/f, H/2, W/2)
        x = self.up4(x, x1)    # (B, 64, H, W)

        # -------------------------
        # 최종 logits 출력
        # -------------------------
        logits = self.outc(x)  # (B, n_classes, H, W)
        return logits


# =========================
# =========================
# 4) 모델 생성 및 구조 확인
# =========================
model = UNet(n_channels=1, n_classes=1).to(device)

# -------------------------
# 4) 모델 구조 확인: print(model) + torchinfo.summary
# -------------------------
print(model)

# torchinfo.summary는 forward를 한 번 실행해 각 레이어의 출력 shape/파라미터 수를 정리해줍니다.
# - 모델이 학습 모드(train)일 때 BatchNorm/Dropout 등이 있을 수 있으니,
#   구조 확인 목적에서는 eval()로 두는 것이 안전합니다.
model.eval()

try:
    from torchinfo import summary

    # 입력 크기: (B, C, H, W)
    # - 여기서는 (1, 1, 128, 128) 텐서를 기준으로 요약합니다.
    # - 다른 해상도로 실험하려면 H,W만 바꾸면 됩니다.
    _summary = summary(
        model,
        input_size=(1, 1, 128, 128),
        depth=4,
        device=device,   # cpu/cuda 환경에 맞춰 동일 디바이스에서 실행
    )
    print(_summary)
except Exception as e:
    print("[torchinfo.summary] 실행 불가:", repr(e))
    print("대응:")
    print("  1) torchinfo 설치 확인: pip install torchinfo")
    print("  2) 입력 크기/채널 수가 모델 기대와 맞는지 확인")
    print("  3) 앞선 forward sanity check가 통과하는지 확인")

# -------------------------
# 5) Forward sanity check (shape 확인)
# -------------------------
dummy = torch.randn(1, 1, 128, 128, device=device)
with torch.no_grad():
    out = model(dummy)

print("Input shape:", tuple(dummy.shape), "Output shape:", tuple(out.shape))



#### 정답

??? 순서: `nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False)`, `nn.ReLU(inplace=True)`, `nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False)`, `nn.ReLU(inplace=True)`, `nn.MaxPool2d(kernel_size=2)`, `nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)`, `torch.cat([x2, x1], dim=1)`, `nn.Conv2d(in_channels, out_channels, kernel_size=1)`

#### 간략 해설

1. `nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False)`: 이미지의 공간 패턴을 channel feature로 추출하는 핵심 CNN layer입니다.
2. `nn.ReLU(inplace=True)`: ReLU activation에 해당하는 핵심 코드로, 이 위치의 입력/출력 흐름을 결정합니다.
3. `nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False)`: 이미지의 공간 패턴을 channel feature로 추출하는 핵심 CNN layer입니다.
4. `nn.ReLU(inplace=True)`: ReLU activation에 해당하는 핵심 코드로, 이 위치의 입력/출력 흐름을 결정합니다.
5. `nn.MaxPool2d(kernel_size=2)`: max pooling에 해당하는 핵심 코드로, 이 위치의 입력/출력 흐름을 결정합니다.
6. `nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)`: decoder에서 feature map 해상도를 키우는 upsampling layer입니다.
7. `torch.cat([x2, x1], dim=1)`: U-Net decoder에서 encoder feature를 이어 붙여 위치 정보를 복원합니다.
8. `nn.Conv2d(in_channels, out_channels, kernel_size=1)`: 이미지의 공간 패턴을 channel feature로 추출하는 핵심 CNN layer입니다.


### 문제 23. 3) 손실 함수 & 옵티마이저 설정
- 원본 파일/셀: `04_Unet.ipynb` / code cell index `8`
- 목적: 목적:** Segmentation 학습에 필요한 손실 함수와 최적화 방법을 설정합니다.
- 관찰 포인트:
  - **관찰 포인트**
  - Binary segmentation이면 보통 `BCEWithLogitsLoss`를 많이 사용(로짓 입력 주의)
  - Multi-class segmentation이면 `CrossEntropyLoss`(타깃은 class index) 사용
  - `sigmoid/softmax`를 **언제 적용해야 하는지**(loss와의 궁합)
- 출제 포인트:
  - 모델 결과와 정답을 이용한 학습
- 빈칸 기준:
  - loss 함수 설정
  - optimizer 설정


In [ ]:
# 손실 함수 및 옵티마이저 설정
criterion = ???  # Sigmoid + Binary Cross Entropy
optimizer = ???

# (옵션) 학습 스케줄러
# 20 에포크(Epoch)마다 학습률을 변경, 기존 학습률에 0.5를 곱합
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

#### 정답

??? 순서: `nn.BCEWithLogitsLoss()`, `torch.optim.Adam(model.parameters(), lr=1e-3)`

#### 간략 해설

1. `nn.BCEWithLogitsLoss()`: loss 함수 설정에 해당하는 핵심 코드로, 이 위치의 입력/출력 흐름을 결정합니다.
2. `torch.optim.Adam(model.parameters(), lr=1e-3)`: optimizer는 gradient를 어떤 규칙과 learning rate로 갱신할지 정합니다.


### 문제 24. 4) 학습 루프
- 원본 파일/셀: `04_Unet.ipynb` / code cell index `10`
- 목적: 목적:** 가장 기본적인 형태의 학습/검증 루프를 실행해, loss가 정상적으로 내려가는지 확인합니다.
- 관찰 포인트:
  - **관찰 포인트**
  - `model.train()` / `model.eval()` 모드 전환
  - `optimizer.zero_grad()` → `loss.backward()` → `optimizer.step()` 순서
  - 검증 단계에서 `torch.no_grad()` 사용 여부
- 출제 포인트:
  - 모델 결과와 정답을 이용한 학습
- 빈칸 기준:
  - train mode
  - gradient 초기화
  - forward pass
  - loss 계산
  - backpropagation
  - parameter update
  - gradient 비활성화
  - eval mode


In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    ???
    running_loss = 0.0
    for imgs, masks in dataloader:
        imgs = imgs.to(device)
        masks = masks.to(device)

        ???
        logits = ???
        loss = ???
        ???
        ???

        running_loss += loss.item() * imgs.size(0)

    epoch_loss = running_loss / len(dataloader.dataset)
    return epoch_loss


???
def evaluate(model, dataloader, criterion, device):
    ???
    running_loss = 0.0
    for imgs, masks in dataloader:
        imgs = imgs.to(device)
        masks = masks.to(device)

        logits = model(imgs)
        loss = criterion(logits, masks)
        running_loss += loss.item() * imgs.size(0)

    epoch_loss = running_loss / len(dataloader.dataset)
    return epoch_loss


num_epochs = 10
train_losses = []
val_losses = []

for epoch in range(1, num_epochs + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss = evaluate(model, val_loader, criterion, device)
    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch [{epoch}/{num_epochs}]  train_loss: {train_loss:.4f}  val_loss: {val_loss:.4f}")

#### 정답

??? 순서: `model.train()`, `optimizer.zero_grad()`, `model(imgs)`, `criterion(logits, masks)`, `loss.backward()`, `optimizer.step()`, `@torch.no_grad()`, `model.eval()`

#### 간략 해설

1. `model.train()`: Dropout/BatchNorm 등을 학습 모드로 전환해 parameter update에 맞게 동작시킵니다.
2. `optimizer.zero_grad()`: PyTorch gradient는 누적되므로 새 step 전에 이전 gradient를 초기화합니다.
3. `model(imgs)`: forward pass에 해당하는 핵심 코드로, 이 위치의 입력/출력 흐름을 결정합니다.
4. `criterion(logits, masks)`: 모델 출력과 정답 label/mask를 비교해 학습 신호인 loss를 계산합니다.
5. `loss.backward()`: loss를 기준으로 각 parameter의 gradient를 계산합니다.
6. `optimizer.step()`: 계산된 gradient를 이용해 모델 parameter를 갱신합니다.
7. `@torch.no_grad()`: 평가/시각화에서는 gradient 저장이 필요 없으므로 메모리와 연산을 줄입니다.
8. `model.eval()`: 평가/추론에서는 Dropout을 끄고 BatchNorm running statistics를 사용해야 합니다.


### 문제 25. 5) 결과 시각화
- 원본 파일/셀: `04_Unet.ipynb` / code cell index `14`
- 목적: 목적:** 모델이 실제로 “분할”을 하고 있는지, 입력/정답/예측을 나란히 비교합니다.
- 관찰 포인트:
  - **관찰 포인트**
  - 출력이 로짓이면 `sigmoid` 후 threshold(예: 0.5)로 마스크를 만드는지
  - GT 마스크와 예측 마스크의 차이가 어디에서 주로 발생하는지(경계/작은 객체/노이즈 등)
  - 데이터가 간단한데도 실패한다면: 모델 출력 채널, loss, target 타입을 다시 점검
- 출제 포인트:
  - 모델 결과와 정답을 이용한 학습
  - 평가 metric 및 예측 후처리
- 빈칸 기준:
  - gradient 비활성화
  - eval mode
  - forward pass
  - sigmoid probability


In [ ]:
???
def visualize_predictions(model, dataloader, device, num_samples=3):
    ???
    imgs, masks = next(iter(dataloader))
    imgs = imgs.to(device)
    masks = masks.to(device)

    logits = ???
    probs = ???
    preds = (probs > 0.5).float()

    imgs = imgs.cpu().numpy()
    masks = masks.cpu().numpy()
    preds = preds.cpu().numpy()

    num_samples = min(num_samples, imgs.shape[0])

    plt.figure(figsize=(9, num_samples * 3))
    for i in range(num_samples):
        # 입력
        plt.subplot(num_samples, 3, 3 * i + 1)
        plt.imshow(imgs[i, 0], cmap='gray')
        plt.title('Input')
        plt.axis('off')

        # GT mask
        plt.subplot(num_samples, 3, 3 * i + 2)
        plt.imshow(masks[i, 0], cmap='gray')
        plt.title('GT Mask')
        plt.axis('off')

        # Pred mask
        plt.subplot(num_samples, 3, 3 * i + 3)
        plt.imshow(preds[i, 0], cmap='gray')
        plt.title('Pred Mask')
        plt.axis('off')

    plt.tight_layout()
    plt.show()


visualize_predictions(model, val_loader, device, num_samples=3)

#### 정답

??? 순서: `@torch.no_grad()`, `model.eval()`, `model(imgs)`, `torch.sigmoid(logits)`

#### 간략 해설

1. `@torch.no_grad()`: 평가/시각화에서는 gradient 저장이 필요 없으므로 메모리와 연산을 줄입니다.
2. `model.eval()`: 평가/추론에서는 Dropout을 끄고 BatchNorm running statistics를 사용해야 합니다.
3. `model(imgs)`: forward pass에 해당하는 핵심 코드로, 이 위치의 입력/출력 흐름을 결정합니다.
4. `torch.sigmoid(logits)`: binary segmentation logit을 0~1 foreground 확률로 바꿉니다.
